# Spécialiser Laya au tri d'extraits RAG en français

**But.** Laya (ConvAI Innovations, Apache-2.0) est un modèle de décision : à partir d'un état structuré et
d'une consigne, il rend la probabilité d'un « oui » (`noul`). Ses auteurs le présentent comme une **base à
spécialiser**. Ce notebook l'entraîne sur nos deux décisions RAG, avec exactement le format qu'envoie le banc :

| Tâche | État | Clé | Étiquette |
|---|---|---|---|
| `pertinence` | question + un des 10 premiers candidats de la recherche | `pertinent` | l'extrait porte au moins 95 % de la citation probante (moins du tiers : faux ; entre les deux : écarté) |
| `reponse_possible` | question + les 7 premiers extraits (~3 000 tokens) | `repondable` | l'un des 7 est pertinent ; faux pour les questions hors corpus |

**Pourquoi.** Au banc R-54 (rapport 43), Laya **sans entraînement** place un extrait pertinent au 1er rang
pour **13 %** des questions, contre **41 % sans reranker** et **46 % avec mmarco** : en l'état, il dégrade la
recherche. La question posée ici est de savoir si un entraînement sur nos données comble cet écart.

**Données publiques seulement.** Le jeu ne contient que le Code des assurances (compilation Droit.org des
données Légifrance) et des questions générées à partir de ses articles. Aucun contrat client, aucune donnée
Docaposte : c'est la condition pour entraîner hors de Docaposte, sur Kaggle.

**Ce que ce notebook ne mesure pas.** L'évaluation ci-dessous porte sur `laya-eval.jsonl` (10 % des
questions d'entraînement, mises de côté). Elle sert à vérifier que l'entraînement apprend quelque chose. La
mesure qui compte reste le banc, sur les jeux `golden/` jamais vus ici : rapatrier le checkpoint et relancer
`run_rag_benchmark.py`.

**Réglages Kaggle.** *Accelerator* : GPU T4 (une carte suffit) ; *Internet* : activé (pip et Hugging Face) ;
jeu de données Laya importé (privé). Lancer de préférence par *Save Version → Save & Run All* : l'entraînement
continue sans navigateur ouvert, et les sorties de `/kaggle/working` sont conservées. Aucune clé n'est
nécessaire : le dépôt du modèle de base est public.

## Sommaire

| # | Section | Ce qu'on y obtient |
|---|---|---|
| 1 | Environnement | GPU vérifié, une seule carte |
| 2 | Installation | `laya==0.4.22`, la version du banc |
| 3 | Paramètres | tous les réglages en un seul endroit |
| 4 | Données | empreintes, comptes et structure contrôlés |
| 5 | Statistiques | volume et équilibre des classes |
| 6 | Checkpoint de base | révision figée du banc, empreinte |
| 7 | Pilote et vérification | pilote d'entraînement (arrêt anticipé, reprise) écrit, exemples vérifiés sans entraîner |
| 8 | Évaluation **avant** | référence du modèle d'origine |
| 9 | Entraînement | meilleur point de validation, calibré ; états de reprise ; journal, durée |
| 10 | Évaluation **après** | tableau avant / après |
| 11 | Visualisation | métriques, distributions, courbes ROC, perte, validation |
| 12 | Conclusion | verdict chiffré, règle de lecture, suite |
| 13 | Archivage | zip, manifeste et empreinte à télécharger |

Chaque section commence par ce qu'elle fait et pourquoi ; les sorties (JSON, CSV, figures, conclusion) sont
écrites dans `/kaggle/working` au fur et à mesure, pour survivre à une coupure du noyau.

## 1. Environnement

On vérifie le GPU **avant** d'installer quoi que ce soit : sans carte, l'entraînement tournerait sur CPU
pendant des jours sans prévenir. Le contrôle de `torch` se fait dans un sous-processus pour ne pas charger
`torch` dans le noyau avant l'installation de `laya` (qui pourrait en changer la version).

In [ ]:
import os
import subprocess
import sys

# Une seule carte : le pilote d'entraînement tourne sur un seul device. Fixer la carte évite qu'une
# bibliothèque répartisse le travail sur les deux T4 sans qu'on l'ait décidé.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if smi.returncode != 0:
    raise RuntimeError("Aucun GPU visible : Settings > Accelerator > GPU T4, puis relancer la session.")
print(smi.stdout)

SONDE_TORCH = (
    "import torch, sys; ok = torch.cuda.is_available(); "
    "print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| disponible', ok); "
    "p = torch.cuda.get_device_properties(0) if ok else None; "
    "print(f'GPU 0 : {p.name}, {p.total_memory / 2**30:.1f} Gio, bf16={torch.cuda.is_bf16_supported()}') if ok else None; "
    "sys.exit(0 if ok else 1)"
)


def verifier_torch():
    r = subprocess.run([sys.executable, "-c", SONDE_TORCH], capture_output=True, text=True)
    print(r.stdout, r.stderr[-2000:], sep="")
    if r.returncode != 0:
        raise RuntimeError("torch ne voit pas CUDA : vérifier l'accélérateur de la session.")


print("Python", sys.version.split()[0])
verifier_torch()
# Le T4 ne gère pas bf16 : si une précision mixte est utilisée, ce sera fp16.

## 2. Installation de Laya, version épinglée

`laya==0.4.22`, la version du conteneur du banc (`benchmarks/rag/laya/Dockerfile`) : le checkpoint produit doit
être relu à l'identique par ce conteneur. L'extra `[serve]` (serveur HTTP) est inutile ici.

In [ ]:
import importlib.metadata as md

torch_avant = md.version("torch")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "laya==0.4.22"], capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-3000:], sep="")
if r.returncode != 0:
    raise RuntimeError("pip install laya==0.4.22 a échoué (Internet activé dans la session ?)")

torch_apres = md.version("torch")
print("laya", md.version("laya"), "| torch", torch_avant, "->", torch_apres)
if torch_apres != torch_avant:
    # Une autre version de torch peut ne pas correspondre au pilote CUDA de Kaggle : on revérifie.
    print("ATTENTION : l'installation de laya a changé la version de torch ; nouvelle vérification de CUDA.")
verifier_torch()

if md.version("laya") != "0.4.22":
    raise RuntimeError(f"laya {md.version('laya')} installé au lieu de 0.4.22")

## 3. Paramètres

Tout ce qui change un résultat est ici, et recopié dans `train-config.json` à la fin de l'entraînement.

**Mémoire.** Sur la Tesla T4, l'essai à 2 exemples par micro-batch et 3 072 tokens a saturé la VRAM (session
Kaggle du 26/11/2026). Les réglages passent à un exemple par micro-batch et 64 étapes d'accumulation : le
batch effectif reste 64, comme les 8 × 8 par défaut de `laya-train` ; on change la mémoire, pas
l'optimisation. `MAX_LEN = 2048` limite la longueur des séquences d'entraînement : les cas
`reponse_possible` (~3 000 tokens) perdent leurs derniers extraits pendant l'entraînement, l'évaluation garde
les longueurs du banc. Si la mémoire manque encore, baisser `MAX_LEN` à 1536. Sur GPU, laya active déjà de
lui-même la précision mixte (fp16) et le point de contrôle de gradient (`laya/train.py`, `amp=None` → actif
sur CUDA) : il n'y a pas d'autre levier mémoire que le micro-batch et la longueur.

**Arrêt anticipé.** 10 % des questions d'entraînement (`PART_VALIDATION`) sont mises de côté, par question,
pour la **validation** — jamais `laya-eval`, qui reste la mesure finale. Elle est évaluée `EVALS_PAR_EPOQUE`
fois par époque ; le score est l'AUROC moyenne des deux tâches (indépendante de la température). Le meilleur
point est gardé ; après `PATIENCE` validations sans progrès d'au moins `DELTA_MIN`, l'entraînement s'arrête.
Le checkpoint final est ce meilleur point, **calibré** sur la même validation.

**Reprise.** Le pilote écrit un état complet (poids, optimiseur, planificateur, générateurs aléatoires,
position dans l'époque) toutes les `SAUVEGARDE_MIN` minutes et à chaque fin d'époque, dans `ETATS_DIR`. Une
reprise refait exactement l'entraînement ininterrompu : vérifié au bit près sur CPU (voir
`tests/test_entrainer_laya.py` du dépôt smartsumup-plus). Sur GPU, deux exécutions ne sont déjà pas
identiques au bit près (calculs non déterministes) : la reprise repart du même état, pas des mêmes arrondis.
Pour reprendre :

- `REPRENDRE = "dernier"` : depuis l'état le plus récent (milieu d'époque compris) ;
- `REPRENDRE = 2` : depuis la fin de l'époque 2, si son état est encore là (les `GARDER_ETATS_EPOQUE` plus
  récents sont gardés, ~3,7 Go chacun) ;
- dans une **nouvelle session** Kaggle : ajouter la sortie de la session précédente comme donnée d'entrée,
  et pointer `DOSSIER_REPRISE` sur son dossier `etats` (sous `/kaggle/input/…`).

À la reprise, on peut changer `EPOCHS`, `PATIENCE` ou le budget ; les réglages qui changent les calculs
(micro-batch, accumulation, taux, longueur, graine, données) doivent rester les mêmes, sinon le pilote refuse.

**Durée.** Kaggle coupe une session au bout de 12 h. La session du 26/11 s'est arrêtée au milieu de la
2ᵉ époque sur 4, et tout ce qui suivait l'entraînement a été perdu. Au-delà de `DUREE_MAX_H`, le pilote
sauvegarde son état, calibre le meilleur point et rend la main : le notebook va au bout, et l'entraînement
pourra être repris. Une estimation du temps restant s'affiche à chaque dixième d'époque.

In [ ]:
import hashlib
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import threading
import time
from pathlib import Path

LAYA_VERSION = "0.4.22"
REPO = "convaiinnovations/laya"
REVISION = "7b928d828b7b0e022f929d9bd2e44165aa270148"   # révision du banc
SUBFOLDER = "multilingual"                               # nos questions sont en français

MAX_LEN = 2048          # limite d'entraînement pour la mémoire du GPU ; 1536 si elle manque encore
HEAD_MAX_LEN = None     # None : valeur par défaut de laya
MICRO_BATCH = 1         # un exemple par passage sur le GPU (2 x 3072 tokens saturait la T4)
GRAD_ACCUM = 64         # batch effectif : 1 x 64 = 64
EPOCHS = 4
ENCODER_LR = 2.5e-5
HEAD_LR = 1e-4
LOSS = "rlcd"
SEED = 0
DUREE_MAX_H = 10.0      # Kaggle coupe à 12 h : garder ~1 h pour l'évaluation après, les figures et l'archive

# Arrêt anticipé et reprise (pilote entrainer_laya.py, section 7).
PART_VALIDATION = 0.1   # part des questions d'entraînement mises de côté pour l'arrêt anticipé et le calibrage
EVALS_PAR_EPOQUE = 2    # validations par époque, la dernière en fin d'époque
PATIENCE = 2            # validations sans progrès avant l'arrêt ; 0 : jamais
DELTA_MIN = 0.002       # progrès minimal de l'AUROC moyenne de validation pour compter
SAUVEGARDE_MIN = 20     # minutes entre deux états de reprise (~3,7 Go, ~30 s d'écriture)
GARDER_ETATS_EPOQUE = 2 # états de fin d'époque conservés : on ne peut reprendre qu'à ces époques-là
REPRENDRE = None        # None : nouvel entraînement ; "dernier" : état le plus récent ; 2 : fin de l'époque 2
DOSSIER_REPRISE = None  # None : ETATS_DIR ; sinon le dossier etats d'une session précédente (/kaggle/input/…)

# Évaluation : mêmes longueurs que le banc (run_rag_benchmark.py : rien pour la pertinence,
# --laya-max-len 4096 pour la réponse possible), pour que « avant » ressemble à ce que le banc a mesuré.
EVAL_MAX_LEN = {"pertinence": None, "reponse_possible": 4096}
EVAL_BATCH = {"pertinence": 32, "reponse_possible": 4}   # divisés par deux automatiquement en cas d'OOM
SEUIL = 0.5

DATA_DIR = None         # None : détection sous /kaggle/input ; sinon chemin du dossier des fichiers
WORK = Path("/kaggle/working")
OUT_DIR = WORK / "laya-rag-fr"
ETATS_DIR = WORK / "etats"
PILOTE = Path("/kaggle/working/entrainer_laya.py")       # écrit par la section 7 (%%writefile)
RESULTAT_TRAIN = WORK / "train-resultat.json"
LOG_DRY_RUN = WORK / "laya-dry-run.log"
LOG_TRAIN = WORK / "laya-train.log"


def sha256_file(path, bloc=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for morceau in iter(lambda: f.read(bloc), b""):
            h.update(morceau)
    return h.hexdigest()


def lire_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def hms(secondes):
    secondes = int(secondes)
    return f"{secondes // 3600}:{secondes // 60 % 60:02d}:{secondes % 60:02d}"


PAS_LAYA = re.compile(r"epoch (\d+)/(\d+) step (\d+) ")


def run_logged(cmd, log_path, *, env_overrides=None, ajouter=False, filet_s=None, budget_s=None,
               pas_par_epoque=None):
    """Lance une commande, affiche sa sortie en direct et la recopie, horodatée, dans un journal.

    Pourquoi un sous-processus : la mémoire GPU de l'entraînement est rendue à sa fin, et le
    journal survit à une coupure du noyau. Rend (durée en s, arrêt) : arrêt vaut None si la
    commande est allée au bout, "interruption" si la cellule a été interrompue, "filet" si
    `filet_s` est dépassé. Dans ces deux cas la commande reçoit SIGTERM : le pilote finit sa
    fenêtre d'accumulation, sauvegarde et calibre, puis rend la main (quelques minutes). Une
    seconde interruption le tue sans attendre. Lève une erreur si la commande échoue d'elle-même.
    `budget_s` et `pas_par_epoque` servent à afficher le temps restant estimé.
    """
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    if env_overrides:
        env.update(env_overrides)
    print("$", shlex.join(cmd), flush=True)
    debut = time.time()
    arret = None
    prochain_point, fait_initial = None, None
    with open(log_path, "a" if ajouter else "w", encoding="utf-8") as log:
        log.write("$ " + shlex.join(cmd) + "\n")
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        minuteurs = []

        def armer(delai, action):
            t = threading.Timer(delai, action)
            t.daemon = True
            t.start()
            minuteurs.append(t)

        def arreter(raison):
            nonlocal arret
            arret = arret or raison
            proc.terminate()
            armer(1800, proc.kill)          # le pilote a 30 min pour sauvegarder et calibrer

        if filet_s:
            # Filet de sécurité : le pilote respecte déjà son budget ; ceci ne sert que s'il se bloque.
            armer(filet_s, lambda: arreter("filet"))

        def lire():
            nonlocal prochain_point, fait_initial
            for ligne in proc.stdout:
                ecoule = time.time() - debut
                print(ligne, end="", flush=True)
                log.write(f"[{hms(ecoule)}] {ligne}")
                log.flush()
                m = PAS_LAYA.match(ligne)
                if not (m and pas_par_epoque):
                    continue
                epoque, total, pas = (int(g) for g in m.groups())
                fait = ((epoque - 1) * pas_par_epoque + pas) / (total * pas_par_epoque)
                if fait_initial is None:   # une reprise ne repart pas de zéro
                    fait_initial, prochain_point = fait, fait + 0.1 / total
                elif fait >= prochain_point and fait > fait_initial:
                    restant = ecoule / (fait - fait_initial) * (1 - fait)
                    alerte = (f" : DÉPASSE le budget de {hms(budget_s)} (le pilote s'arrêtera à temps ; "
                              "baisser EPOCHS ou prévoir une reprise)" if budget_s and ecoule + restant > budget_s else "")
                    point = f"# {100 * fait:.1f} % en {hms(ecoule)} pour cette session, reste estimé {hms(restant)}{alerte}"
                    print(point, flush=True)
                    log.write(point + "\n")
                    prochain_point += 0.1 / total

        try:
            try:
                lire()
            except KeyboardInterrupt:
                print("\nArrêt demandé : le pilote finit sa fenêtre d'accumulation, sauvegarde et calibre. "
                      "Interrompre une seconde fois pour le tuer sans attendre.", flush=True)
                arreter("interruption")
                lire()                       # suite de la sortie jusqu'à la fin du pilote
            code = proc.wait()
        except KeyboardInterrupt:
            proc.kill()
            code = proc.wait()
            arret = "tue"
        finally:
            for t in minuteurs:
                t.cancel()
        duree = time.time() - debut
        log.write(f"\n# code de retour {code}, durée {duree:.0f} s, arrêt : {arret or 'aucun'}\n")
    if arret:
        print(f"\n{Path(cmd[1] if len(cmd) > 1 else cmd[0]).name} arrêté ({arret}) après {hms(duree)} ; "
              f"journal : {log_path}", flush=True)
    elif code != 0:
        raise RuntimeError(f"{shlex.join(cmd[:2])} a échoué (code {code}) ; journal : {log_path}")
    return duree, arret


WORK.mkdir(parents=True, exist_ok=True)
print("Sorties dans", WORK)

## 4. Localisation des données et contrôle des empreintes

Le nom du jeu Kaggle n'est pas connu d'avance : on cherche `laya-dataset.meta.json` sous `/kaggle/input`. On
refuse d'entraîner si un fichier diffère de ce que le constructeur (`build_laya_dataset.py`) a écrit :
empreinte sha256, nombre de lignes, compagnon `.meta.jsonl` de même longueur, structure de chaque ligne. Un
entraînement sur un fichier tronqué ou mal importé donnerait un résultat impossible à relier au banc.

In [ ]:
if DATA_DIR is None:
    trouves = sorted(Path("/kaggle/input").glob("**/laya-dataset.meta.json"))
    if len(trouves) != 1:
        raise RuntimeError(f"{len(trouves)} fichier(s) laya-dataset.meta.json sous /kaggle/input : {trouves} ; "
                           "importer le jeu, ou fixer DATA_DIR à la main.")
    DATA_DIR = trouves[0].parent
DATA_DIR = Path(DATA_DIR)
META = json.loads((DATA_DIR / "laya-dataset.meta.json").read_text(encoding="utf-8"))
print("Données :", DATA_DIR)

CLE_PAR_TACHE = {"pertinence": "pertinent", "reponse_possible": "repondable"}
CHAMPS_ETAT = {"pertinence": {"question", "passage"}, "reponse_possible": {"question", "passages"}}

erreurs = []
FICHIERS = {}
for split in ("entrainement", "evaluation"):
    attendu = META["fichiers"][split]
    path = DATA_DIR / attendu["nom"]
    compagnon = path.with_suffix(".meta.jsonl")   # laya-train.jsonl -> laya-train.meta.jsonl
    FICHIERS[split] = (path, compagnon)
    for p in (path, compagnon):
        if not p.exists():
            erreurs.append(f"absent : {p}")
    if not path.exists() or not compagnon.exists():
        continue
    empreinte = sha256_file(path)
    if empreinte != attendu["sha256"]:
        erreurs.append(f"{path.name} : sha256 {empreinte} au lieu de {attendu['sha256']}")
    cas, metas = lire_jsonl(path), lire_jsonl(compagnon)
    if len(cas) != attendu["lignes"]:
        erreurs.append(f"{path.name} : {len(cas)} lignes au lieu de {attendu['lignes']}")
    if len(metas) != len(cas):
        erreurs.append(f"{compagnon.name} : {len(metas)} lignes pour {len(cas)} cas")
    for i, (c, m) in enumerate(zip(cas, metas)):
        tache = m.get("tache")
        cle = CLE_PAR_TACHE.get(tache)
        if (cle is None or set(c) != {"state", "questions", "expected"} or set(c["questions"]) != {cle}
                or c["questions"][cle].get("type") != "noul" or not isinstance(c["expected"].get(cle), bool)
                or not CHAMPS_ETAT[tache] <= set(c["state"])):
            erreurs.append(f"{path.name} ligne {i + 1} : structure inattendue pour la tâche {tache!r}")
            if len(erreurs) > 20:
                break
    print(f"{split:12s} {path.name:20s} {len(cas):6d} cas  sha256 {empreinte[:16]}…")

if erreurs:
    raise RuntimeError("Jeu refusé :\n- " + "\n- ".join(erreurs))

TRAIN, TRAIN_META = FICHIERS["entrainement"]
EVAL, EVAL_META = FICHIERS["evaluation"]
print("Empreintes, comptes et structure conformes à laya-dataset.meta.json.")
print("Graine du découpage :", META.get("graine_decoupage"), "| part évaluation :", META.get("part_evaluation"))
print("Jeux réservés à la mesure (exclus) :", META.get("jeux_reserves_a_l_evaluation"))

## 5. Statistiques du jeu

Le taux de positifs dit ce que vaut une exactitude : si 85 % des extraits sont « non pertinents », un modèle
qui répond toujours non atteint 85 %. C'est pourquoi on regarde aussi l'AUROC et le Hit@1 plus bas. Les comptes
sont recoupés avec ceux du constructeur.

In [ ]:
from collections import Counter, defaultdict
from statistics import median

for split, (path, compagnon) in FICHIERS.items():
    cas, metas = lire_jsonl(path), lire_jsonl(compagnon)
    comptes = Counter()
    questions = defaultdict(set)
    positifs_par_q = defaultdict(int)
    longueurs = []
    for c, m in zip(cas, metas):
        tache, cle = m["tache"], CLE_PAR_TACHE[m["tache"]]
        y = c["expected"][cle]
        comptes[f"{tache}_{split}_{'vrai' if y else 'faux'}"] += 1
        questions[tache].add(m["question_id"])
        if tache == "pertinence":
            positifs_par_q[m["question_id"]] += int(y)
        else:
            longueurs.append(len(json.dumps(c["state"], ensure_ascii=False)))
    print(f"== {split}")
    for tache in CLE_PAR_TACHE:
        v, f = comptes[f"{tache}_{split}_vrai"], comptes[f"{tache}_{split}_faux"]
        n = v + f
        print(f"  {tache:17s} {n:6d} cas, {len(questions[tache]):5d} questions, positifs {v} ({v / max(n, 1):.1%})")
    print(f"  questions avec au moins un extrait pertinent : {sum(1 for x in positifs_par_q.values() if x)}"
          f" / {len(positifs_par_q)}")
    if longueurs:
        # Ordre de grandeur seulement : ~3 à 4 caractères par token en français.
        print(f"  reponse_possible, état sérialisé : médiane {median(longueurs):.0f} car., max {max(longueurs)} car.")
    attendus = {k: v for k, v in META.get("comptes", {}).items() if k.startswith(("pertinence_", "reponse_possible_")) and f"_{split}_" in k}
    ecarts = {k: (comptes.get(k, 0), attendus.get(k, 0)) for k in set(attendus) | set(comptes)
              if comptes.get(k, 0) != attendus.get(k, 0)}
    if ecarts:
        raise RuntimeError(f"Comptes différents de laya-dataset.meta.json (calculé, attendu) : {ecarts}")

## 6. Checkpoint de base, à la révision du banc

Laisser laya télécharger `multilingual` prendrait la dernière version du dépôt, sans révision fixe : l'écart
avant / après pourrait alors venir d'un changement de poids en amont. On télécharge donc la révision épinglée
(celle du conteneur du banc) et on passe le chemin local du sous-dossier `multilingual/`.

In [ ]:
from huggingface_hub import snapshot_download

# Mêmes motifs que le Dockerfile du banc (*.py compris), pour obtenir exactement les mêmes fichiers.
snapshot = Path(snapshot_download(REPO, revision=REVISION, allow_patterns=[f"{SUBFOLDER}/*", "config.json", "*.py"]))
BASE_DIR = snapshot / SUBFOLDER
manquants = [n for n in ("rl_agent_config.json", "model.safetensors", "tokenizer", "encoder") if not (BASE_DIR / n).exists()]
if manquants:
    raise RuntimeError(f"Checkpoint de base incomplet dans {BASE_DIR} : {manquants}")

BASE_INFO = {
    "depot": REPO, "revision": REVISION, "sous_dossier": SUBFOLDER, "chemin": str(BASE_DIR),
    "model_safetensors_sha256": sha256_file(BASE_DIR / "model.safetensors"),
    "fichiers": sorted(str(p.relative_to(BASE_DIR)) for p in BASE_DIR.rglob("*") if p.is_file()),
}
(WORK / "base-checkpoint.json").write_text(json.dumps(BASE_INFO, ensure_ascii=False, indent=1), encoding="utf-8")
taille = sum(p.stat().st_size for p in BASE_DIR.rglob("*") if p.is_file())
print(f"Base : {BASE_DIR} ({taille / 2**20:.0f} Mio)")
print("model.safetensors sha256 :", BASE_INFO["model_safetensors_sha256"])

## 7. Pilote d'entraînement et vérification à blanc

`laya-train` (laya 0.4.22) ne sauvegarde qu'en fin d'époque, sans l'état de l'optimiseur, ne calibre qu'après
la dernière époque et n'a pas d'arrêt anticipé : la session du 26/11 a perdu tout ce qui suivait la coupure. Le
**pilote** ci-dessous appelle les mêmes fonctions de `laya.train` (mêmes exemples, même perte, même optimiseur,
même planificateur, même boucle que `train_model`) et y ajoute la validation, l'arrêt anticipé, les états de
reprise et le budget de durée. Son checkpoint se charge avec `laya.load`, comme celui de `laya-train`.

La première cellule l'écrit dans `/kaggle/working`. Il est versionné avec ses tests dans le dépôt
smartsumup-plus (`benchmarks/rag/laya/entrainement/entrainer_laya.py`) : le modifier là, puis régénérer ce
notebook.

La seconde lance `--verifier` : construction et découpage des exemples, sans entraîner. Une erreur de format
coûte quelques secondes ici au lieu d'une session GPU. La commande est exactement celle de l'entraînement,
plus `--verifier`.

In [ ]:
%%writefile /kaggle/working/entrainer_laya.py
"""Entraîner Laya avec arrêt anticipé et reprise exacte (pilote du notebook Kaggle).

Pourquoi ce pilote plutôt que `laya-train` : la commande de laya 0.4.22 ne sauvegarde qu'en fin
d'époque, sans l'état de l'optimiseur, ne calibre qu'après la dernière époque et n'a pas d'arrêt
anticipé. Le 26/11/2026, une session Kaggle arrêtée au milieu de la 2e époque sur 4 a laissé un
checkpoint non calibré et rien pour reprendre. Ce pilote appelle les fonctions de `laya.train`
(mêmes exemples, même perte, même optimiseur, même planificateur, même boucle que `train_model`)
et ajoute :

- une **validation par question**, tirée du jeu d'entraînement (jamais de laya-eval, qui reste la
  mesure finale) : elle sert à l'arrêt anticipé, puis au calibrage des températures ;
- un **état de reprise complet** (poids, optimiseur, planificateur, échelle fp16, générateurs
  aléatoires, position dans l'époque), écrit à intervalle régulier et à chaque fin d'époque : une
  reprise refait exactement l'entraînement ininterrompu ;
- un **budget de durée**, et l'arrêt sur SIGTERM : l'état est sauvegardé, puis le meilleur point
  de validation est calibré et écrit, au lieu d'être perdu avec la session.

Le checkpoint écrit dans `--out` se charge avec `laya.load`, comme celui de `laya-train`.
Ce pilote dépend de fonctions internes de laya : la version est vérifiée au démarrage.

    python entrainer_laya.py --data laya-train.jsonl --meta laya-train.meta.jsonl --base <base> \\
        --out laya-rag-fr --etats etats --resultat train-resultat.json [--reprendre dernier|N]
"""
from __future__ import annotations

import argparse
import hashlib
import importlib.metadata as md
import json
import math
import os
import random
import shutil
import signal
import sys
import time
import warnings
from dataclasses import asdict
from pathlib import Path

LAYA_VERSION = "0.4.22"
if md.version("laya") != LAYA_VERSION:
    raise SystemExit(f"laya {md.version('laya')} installé : ce pilote suit les fonctions internes de laya {LAYA_VERSION}")

import torch  # noqa: E402
from laya.calibrate import fit_abstention_thresholds, fit_temperature_map  # noqa: E402
from laya.common import collate_items, uses_parallel_layout  # noqa: E402
from laya.train import (  # noqa: E402
    TrainConfig, _forward, calibration_records, calibration_report, encode_item, evaluate_records,
    items_from_rows, load_checkpoint, optimizer_updates, prior_collapse_message, read_jsonl,
    resolve_device, rlcd_loss, save_checkpoint, sigma_at, soft_ce_loss, update_budget_message,
)

TACHES = ("pertinence", "reponse_possible")
# Réglages qui changent la suite des calculs : une reprise doit les retrouver à l'identique.
FIGES = ("micro_batch", "grad_accum", "encoder_lr", "head_lr", "loss", "max_len", "head_max_len", "seed",
         "part_validation", "graine_validation", "donnees_sha256")


def arguments(argv=None):
    p = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    p.add_argument("--data", required=True, help="laya-train.jsonl")
    p.add_argument("--meta", required=True, help="compagnon .meta.jsonl (tâche et question_id de chaque ligne)")
    p.add_argument("--base", required=True, help="checkpoint de départ (dossier au format laya.load)")
    p.add_argument("--out", required=True, help="checkpoint final, calibré")
    p.add_argument("--etats", required=True, help="dossier des états de reprise et du meilleur point")
    p.add_argument("--resultat", required=True, help="résumé JSON de l'entraînement")
    p.add_argument("--reprendre", default=None, help="'dernier' (état le plus récent) ou N (fin de l'époque N)")
    p.add_argument("--dossier-reprise", default=None,
                   help="où lire l'état à reprendre (défaut : --etats ; par ex. la sortie d'une session précédente)")
    p.add_argument("--epochs", type=int, default=4)
    p.add_argument("--micro-batch", type=int, default=8)
    p.add_argument("--grad-accum", type=int, default=8)
    p.add_argument("--encoder-lr", type=float, default=2.5e-5)
    p.add_argument("--head-lr", type=float, default=1e-4)
    p.add_argument("--loss", default="rlcd", choices=["rlcd", "soft-ce"])
    p.add_argument("--max-len", type=int, default=None)
    p.add_argument("--head-max-len", type=int, default=None)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--part-validation", type=float, default=0.1, help="part des questions mises de côté")
    p.add_argument("--graine-validation", default="laya-validation-2026-11-26")
    p.add_argument("--evals-par-epoque", type=int, default=2, help="validations par époque (la dernière en fin d'époque)")
    p.add_argument("--patience", type=int, default=2, help="validations sans progrès avant l'arrêt (0 : jamais)")
    p.add_argument("--delta-min", type=float, default=0.002, help="progrès minimal du score pour compter")
    p.add_argument("--duree-max-h", type=float, default=None, help="budget de cette session, en heures")
    p.add_argument("--sauvegarde-min", type=float, default=20.0, help="intervalle entre deux états de reprise")
    p.add_argument("--garder-etats-epoque", type=int, default=2, help="états de fin d'époque conservés (~3,7 Go chacun)")
    p.add_argument("--lot-validation", type=int, default=4)
    p.add_argument("--log-every", type=int, default=100)
    p.add_argument("--device", default="auto")
    p.add_argument("--verifier", action="store_true", help="construire et découper les exemples, sans entraîner")
    p.add_argument("--arret-apres-updates", type=int, default=None, help=argparse.SUPPRESS)  # tests de reprise
    return p.parse_args(argv)


def sha256_file(path, bloc=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for morceau in iter(lambda: f.read(bloc), b""):
            h.update(morceau)
    return h.hexdigest()


def en_validation(question_id, part, graine):
    """Affectation stable par question : jamais une même question des deux côtés."""
    h = hashlib.sha256(f"{graine}:{question_id}".encode()).hexdigest()
    return int(h[:8], 16) / 0xFFFFFFFF < part


def auroc(etiquettes, scores):
    """AUROC par la statistique de Mann-Whitney (rangs moyens en cas d'égalité) ; None si une classe manque."""
    n_pos = sum(etiquettes)
    n_neg = len(etiquettes) - n_pos
    if not n_pos or not n_neg:
        return None
    ordre = sorted(range(len(scores)), key=scores.__getitem__)
    rangs = [0.0] * len(scores)
    i = 0
    while i < len(ordre):
        j = i
        while j + 1 < len(ordre) and scores[ordre[j + 1]] == scores[ordre[i]]:
            j += 1
        for k in range(i, j + 1):
            rangs[ordre[k]] = (i + j) / 2 + 1
        i = j + 1
    return (sum(r for r, y in zip(rangs, etiquettes) if y) - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def proba_oui(logits):
    """Probabilité du « oui » d'une question noul : l'option 1 (laya.common.render_options : [false, true])."""
    m = max(float(v) for v in logits)
    e = [math.exp(float(v) - m) for v in logits]
    return e[1] / sum(e)


def records(model, tok, items, dev, max_len, head_max_len, parallel, lot):
    # calibration_records ne coupe pas le calcul du gradient : sans no_grad, la mémoire du graphe
    # de 2 048 tokens par exemple s'ajouterait à celle de l'entraînement.
    with torch.no_grad():
        return calibration_records(model, tok, items, dev, max_len, head_max_len, batch_size=lot, parallel=parallel)


def valider(model, tok, validation, dev, max_len, head_max_len, parallel, lot):
    """AUROC par tâche (indépendante de la température) et perte, sur la validation."""
    par_tache, tous = {}, []
    for tache, items in validation.items():
        if not items:
            continue
        recs = records(model, tok, items, dev, max_len, head_max_len, parallel, lot)
        tous += recs
        par_tache[tache] = {"cas": len(recs), "positifs": sum(1 for r in recs if r[2][1] > 0.5),
                            "auroc": auroc([r[2][1] > 0.5 for r in recs], [proba_oui(r[1]) for r in recs]),
                            "perte": round(evaluate_records(recs)["loss"], 5)}
    model.train()
    aurocs = [v["auroc"] for v in par_tache.values() if v["auroc"] is not None]
    perte = evaluate_records(tous)["loss"] if tous else None
    # Score de l'arrêt anticipé : la moyenne des AUROC, ce que le banc attend d'un tri. La perte
    # dépend de la température, que seul le calibrage final règle : elle n'est que rapportée.
    score = sum(aurocs) / len(aurocs) if aurocs else (-perte if perte is not None else None)
    return {"score": score, "perte": perte, "taches": par_tache}


class Etats:
    """États de reprise : `courant.pt` (le plus récent) et `epoque-N.pt` (fin de l'époque N).

    Chaque état a un compagnon JSON lisible sans charger les 3,7 Go de tenseurs. Écriture
    atomique (fichier temporaire puis renommage) : une coupure pendant l'écriture laisse
    l'état précédent intact.
    """

    def __init__(self, dossier, garder):
        self.dossier = Path(dossier)
        self.garder = garder
        self.dossier.mkdir(parents=True, exist_ok=True)

    def epoques(self, dossier=None):
        d = Path(dossier or self.dossier)
        return sorted((int(p.stem.split("-")[1]), p) for p in d.glob("epoque-*.pt"))

    @staticmethod
    def resume(chemin):
        return json.loads(Path(chemin).with_suffix(".json").read_text(encoding="utf-8"))

    def existants(self):
        return sorted(self.dossier.glob("*.pt"))

    def supprimer(self, chemin):
        for p in (Path(chemin), Path(chemin).with_suffix(".json")):
            p.unlink(missing_ok=True)

    def sauver(self, nom, model, optimizer, scheduler, scaler, etat):
        chemin = self.dossier / f"{nom}.pt"
        if nom.startswith("epoque-"):
            # Place pour le nouvel état : on ne garde que les `garder - 1` époques les plus récentes
            # (au moins la nouvelle survit, même avec garder=0 : c'est alors le seul point de reprise).
            anciennes = [p for _, p in self.epoques()]
            for p in anciennes[: max(0, len(anciennes) - (max(self.garder, 1) - 1))]:
                self.supprimer(p)
        taille = sum(t.numel() * t.element_size() for t in model.state_dict().values()) * 3   # poids + 2 moments Adam
        libre = shutil.disk_usage(self.dossier).free
        if libre < 1.1 * taille:
            print(f"ATTENTION : {libre / 2**30:.1f} Gio libres pour un état de ~{taille / 2**30:.1f} Gio : "
                  f"{nom} non sauvegardé", flush=True)
            return None
        debut = time.time()
        tmp = chemin.with_name(chemin.name + ".tmp")
        torch.save({"version": 1, "etat": etat, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict() if scaler else None,
                    "rng_torch": torch.get_rng_state(),
                    "rng_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}, tmp)
        os.replace(tmp, chemin)
        tmp_json = chemin.with_name(chemin.stem + ".json.tmp")
        tmp_json.write_text(json.dumps(etat, ensure_ascii=False, indent=1), encoding="utf-8")
        os.replace(tmp_json, chemin.with_suffix(".json"))
        print(f"état sauvegardé : {chemin.name} ({etat['epoch']} époque(s) terminée(s), pas {etat['n_steps']}, "
              f"update {etat['updates']}, {chemin.stat().st_size / 2**30:.2f} Gio, {time.time() - debut:.0f} s)",
              flush=True)
        return chemin


def choisir_reprise(source, reprendre, etats):
    if reprendre == "dernier":
        candidats = [p for p in Path(source).glob("*.pt") if p.with_suffix(".json").exists()]
        if not candidats:
            raise SystemExit(f"--reprendre dernier : aucun état dans {source}")
        return max(candidats, key=lambda p: (Etats.resume(p)["updates"], p.name.startswith("epoque-")))
    try:
        n = int(reprendre)
    except ValueError:
        raise SystemExit(f"--reprendre {reprendre!r} : attendu 'dernier' ou un numéro d'époque") from None
    chemin = Path(source) / f"epoque-{n}.pt"
    if not chemin.exists():
        dispo = [e for e, _ in etats.epoques(source)]
        raise SystemExit(f"--reprendre {n} : {chemin} absent ; époques disponibles : {dispo}")
    return chemin


def main(argv=None):
    args = arguments(argv)
    config = TrainConfig(epochs=args.epochs, micro_batch=args.micro_batch, grad_accum=args.grad_accum,
                         encoder_lr=args.encoder_lr, head_lr=args.head_lr, loss=args.loss, seed=args.seed,
                         max_len=args.max_len, head_max_len=args.head_max_len, log_every=args.log_every)
    config.validate()
    dev = resolve_device(args.device)
    model, tok, cfg = load_checkpoint(args.base)
    model.to(dev)
    max_len = config.max_len or cfg.get("max_len", 512)
    head_max_len = config.head_max_len or cfg.get("head_max_len", 192)
    parallel = uses_parallel_layout(cfg)

    # Découpage par question, avant la construction des exemples : la validation ne partage
    # aucune question avec l'entraînement (les extraits d'une même question se ressemblent).
    lignes, metas = read_jsonl(args.data), read_jsonl(args.meta)
    if len(lignes) != len(metas):
        raise SystemExit(f"{args.meta} : {len(metas)} lignes pour {len(lignes)} dans {args.data}")
    groupes = {}
    for ligne, meta in zip(lignes, metas):
        cote = "validation" if en_validation(meta["question_id"], args.part_validation, args.graine_validation) \
            else "entrainement"
        groupes.setdefault((cote, meta["tache"]), []).append(ligne)
    train_items, validation, ignores = [], {}, {}
    for tache in TACHES:
        items, sk = items_from_rows(tok, groupes.get(("entrainement", tache), []), max_len, head_max_len)
        train_items += items
        validation[tache], sk_v = items_from_rows(tok, groupes.get(("validation", tache), []), max_len, head_max_len)
        for k, v in list(sk.items()) + list(sk_v.items()):
            ignores[k] = ignores.get(k, 0) + v
    if not train_items:
        raise SystemExit(f"{args.data} : aucun exemple d'entraînement (ignorés : {ignores})")
    pas_par_epoque = math.ceil(len(train_items) / config.micro_batch)
    print("train items %d, validation items %d (%s), skipped %r, device %s"
          % (len(train_items), sum(len(v) for v in validation.values()),
             ", ".join(f"{t} {len(v)}" for t, v in validation.items()), ignores, dev), flush=True)
    print(update_budget_message(len(train_items), config), flush=True)
    print(f"validations : {args.evals_par_epoque} par époque, patience {args.patience}, "
          f"delta minimal {args.delta_min} ; budget : {args.duree_max_h or 'aucun'} h", flush=True)
    if args.verifier:
        return 0

    figes = {"micro_batch": config.micro_batch, "grad_accum": config.grad_accum, "encoder_lr": config.encoder_lr,
             "head_lr": config.head_lr, "loss": config.loss, "max_len": max_len, "head_max_len": head_max_len,
             "seed": config.seed, "part_validation": args.part_validation,
             "graine_validation": args.graine_validation,
             "donnees_sha256": {"data": sha256_file(args.data), "meta": sha256_file(args.meta)}}

    # Mise en place identique à laya.train.train_model (0.4.22).
    amp = (dev.type == "cuda") if config.amp is None else bool(config.amp)
    checkpointing = amp if config.gradient_checkpointing is None else bool(config.gradient_checkpointing)
    if checkpointing and hasattr(model.encoder, "gradient_checkpointing_enable"):
        model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = checkpointing
    model.to(dev).train()
    groups = [{"params": [p for n, p in model.named_parameters() if n.startswith("encoder.") and p.requires_grad],
               "lr": config.encoder_lr},
              {"params": [p for n, p in model.named_parameters() if not n.startswith("encoder.") and p.requires_grad],
               "lr": config.head_lr}]
    optimizer = torch.optim.AdamW(groups, weight_decay=config.weight_decay)
    total_updates = optimizer_updates(len(train_items), config.micro_batch, config.grad_accum, config.epochs)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_updates, eta_min=config.min_lr)
    scaler = torch.amp.GradScaler("cuda") if amp and dev.type == "cuda" else None
    params = [p for g in groups for p in g["params"]]

    etats = Etats(args.etats, args.garder_etats_epoque)
    meilleur_dir = etats.dossier / "meilleur"
    reprise = None
    if args.reprendre is None:
        if etats.existants():
            raise SystemExit(f"{etats.dossier} contient déjà des états {[p.name for p in etats.existants()]} : "
                             "--reprendre dernier pour continuer, ou vider le dossier pour repartir de zéro")
        torch.manual_seed(config.seed)
        etat = {"epoch": 0, "debut_lot": 0, "n_steps": 0, "total": 0.0, "updates": 0, "prochaine_eval": 1,
                "historique": [], "evaluations": [], "meilleur": None, "sans_progres": 0, "secondes": 0.0,
                "figes": figes, "epochs": config.epochs, "sessions": []}
        if meilleur_dir.exists():
            shutil.rmtree(meilleur_dir)
    else:
        source = Path(args.dossier_reprise or args.etats)
        chemin = choisir_reprise(source, args.reprendre, etats)
        print(f"reprise depuis {chemin}", flush=True)
        sauve = torch.load(chemin, map_location="cpu", weights_only=True)   # tenseurs et types simples seulement
        etat = sauve["etat"]
        ecarts = {k: (etat["figes"].get(k), figes[k]) for k in FIGES if etat["figes"].get(k) != figes[k]}
        if ecarts:
            raise SystemExit(f"reprise refusée, réglages différents de l'état sauvegardé : {ecarts}")
        model.load_state_dict(sauve["model"])
        optimizer.load_state_dict(sauve["optimizer"])
        scheduler.load_state_dict(sauve["scheduler"])
        if scaler is not None and sauve["scaler"]:
            scaler.load_state_dict(sauve["scaler"])
        torch.set_rng_state(sauve["rng_torch"])
        if sauve["rng_cuda"] is not None and torch.cuda.is_available():
            torch.cuda.set_rng_state_all(sauve["rng_cuda"])
        del sauve
        if etat["epochs"] != config.epochs:
            # Le planificateur cosinus a été tracé pour l'ancien nombre d'époques : on le prolonge.
            scheduler.T_max = total_updates
            warnings.warn(f"époques {etat['epochs']} -> {config.epochs} : la décroissance du taux d'apprentissage "
                          "est retracée ; l'entraînement n'est plus celui d'une traite", RuntimeWarning)
            etat["epochs"] = config.epochs
        # Meilleur point : ses poids doivent être ceux que l'état désigne, sinon ils viennent d'un
        # futur abandonné (reprise à une époque antérieure) ou d'une autre session.
        m = etat.get("meilleur")
        sources_meilleur = [source / "meilleur", meilleur_dir]
        trouve = None
        for d in sources_meilleur:
            pos = d / "position.json"
            if m and pos.exists() and json.loads(pos.read_text(encoding="utf-8")).get("updates") == m["updates"]:
                trouve = d
                break
        if m and trouve is None:
            print(f"meilleur point (update {m['updates']}) introuvable : il sera rétabli aux validations suivantes",
                  flush=True)
            etat["meilleur"], etat["sans_progres"] = None, 0
        elif trouve is not None and trouve.resolve() != meilleur_dir.resolve():
            if meilleur_dir.exists():
                shutil.rmtree(meilleur_dir)
            shutil.copytree(trouve, meilleur_dir)
        if Path(source).resolve() == etats.dossier.resolve():
            # États postérieurs au point de reprise : périmés, ils seraient repris par erreur plus tard.
            for p in etats.existants():
                if p.resolve() != chemin.resolve() and Etats.resume(p)["updates"] > etat["updates"]:
                    print(f"état postérieur au point de reprise supprimé : {p.name}", flush=True)
                    etats.supprimer(p)
        reprise = {"depuis": str(chemin), "updates": etat["updates"], "epoque": etat["epoch"]}

    session = {"debut": time.strftime("%Y-%m-%dT%H:%M:%S%z"), "reprise": reprise}
    etat["sessions"].append(session)
    debut_session = time.time()
    secondes_avant = etat["secondes"]
    derniere_sauvegarde = time.time()
    updates_session = 0
    arret = None
    demande = {"signal": None}

    def sur_signal(numero, _frame):
        # Arrêt demandé de l'extérieur (bouton Interrupt du notebook) : on finit la fenêtre
        # d'accumulation en cours, puis on sauvegarde et on calibre.
        demande["signal"] = signal.Signals(numero).name

    signal.signal(signal.SIGTERM, sur_signal)
    signal.signal(signal.SIGINT, sur_signal)

    def secondes():
        return secondes_avant + (time.time() - debut_session)

    def point_de_validation(epoque, pas):
        res = valider(model, tok, validation, dev, max_len, head_max_len, parallel, args.lot_validation)
        res.update(epoque=epoque, pas=pas, updates=etat["updates"], position=round(epoque - 1 + pas / pas_par_epoque, 4))
        m = etat["meilleur"]
        progres = res["score"] is not None and (m is None or res["score"] > m["score"] + args.delta_min)
        if progres:
            etat["meilleur"] = {k: res[k] for k in ("score", "epoque", "pas", "updates", "position")}
            etat["sans_progres"] = 0
            tmp = meilleur_dir.with_name("meilleur.tmp")
            if tmp.exists():
                shutil.rmtree(tmp)
            save_checkpoint(model, tok, dict(cfg, max_len=max_len, head_max_len=head_max_len), str(tmp))
            (tmp / "position.json").write_text(json.dumps(etat["meilleur"], indent=1), encoding="utf-8")
            if meilleur_dir.exists():
                shutil.rmtree(meilleur_dir)
            os.replace(tmp, meilleur_dir)
        else:
            etat["sans_progres"] += 1
        res["meilleur"] = progres
        etat["evaluations"].append(res)
        detail = ", ".join(f"{t} auroc {v['auroc']:.4f}" if v["auroc"] is not None else f"{t} auroc —"
                           for t, v in res["taches"].items())
        suite = "meilleur" if progres else f"sans progrès {etat['sans_progres']}/{args.patience or '∞'}"
        score = "—" if res["score"] is None else f"{res['score']:.4f}"
        perte = "—" if res["perte"] is None else f"{res['perte']:.4f}"
        print(f"validation époque {epoque}/{config.epochs} pas {pas} : score {score} ({detail}), perte {perte} — {suite}",
              flush=True)
        if args.patience and etat["sans_progres"] >= args.patience:
            return "arret_anticipe"
        return None

    def raison_de_s_arreter():
        if demande["signal"]:
            return "interruption"
        if args.duree_max_h and time.time() - debut_session > args.duree_max_h * 3600:
            return "budget"
        if args.arret_apres_updates is not None and updates_session >= args.arret_apres_updates:
            return "budget"
        return None

    epoch = etat["epoch"]
    while epoch < config.epochs and arret is None:
        epoch_items = list(train_items)
        random.Random(config.seed + epoch).shuffle(epoch_items)
        sigma = sigma_at(epoch, config.epochs, config.sigma_start, config.sigma_end)
        n_steps, total = etat["n_steps"], etat["total"]
        optimizer.zero_grad(set_to_none=True)
        fin_epoque = True
        for start in range(etat["debut_lot"], len(epoch_items), config.micro_batch):
            chunk = [encode_item(tok, it, max_len, head_max_len, None, parallel)
                     for it in epoch_items[start:start + config.micro_batch]]
            batch = collate_items([chunk], tok.pad_token_id)
            logits = _forward(model, batch, dev, amp, False)
            mask = batch["marker_mask"].to(dev)
            target = batch["target"].to(dev)
            if config.loss == "rlcd":
                loss = rlcd_loss(logits, target, mask, batch["qtype"].to(dev), sigma,
                                 config.rl_samples, config.w_sph, config.w_rps)
            else:
                loss = soft_ce_loss(logits, target, mask)
            window_start = (n_steps // config.grad_accum) * config.grad_accum
            window_size = min(config.grad_accum, pas_par_epoque - window_start)
            scaled = loss / window_size
            if scaler is not None:
                scaler.scale(scaled).backward()
            else:
                scaled.backward()
            n_steps += 1
            dernier = start + config.micro_batch >= len(epoch_items)
            fenetre_finie = n_steps % config.grad_accum == 0 or dernier
            if fenetre_finie:
                if scaler is not None:
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(params, config.grad_clip)
                if scaler is not None:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                etat["updates"] += 1
                updates_session += 1
            total += loss.item()
            if config.log_every and n_steps % config.log_every == 0:
                print("epoch %d/%d step %d loss %.4f" % (epoch + 1, config.epochs, n_steps, loss.item()), flush=True)
            if not fenetre_finie or dernier:
                continue
            # Frontière de fenêtre : aucun gradient en cours, l'état est complet et sauvegardable.
            etat.update(debut_lot=start + config.micro_batch, n_steps=n_steps, total=total)
            k = etat["prochaine_eval"]
            if k < args.evals_par_epoque and n_steps >= k * pas_par_epoque / args.evals_par_epoque:
                etat["prochaine_eval"] = k + 1
                arret = point_de_validation(epoch + 1, n_steps)
            arret = arret or raison_de_s_arreter()
            if arret or time.time() - derniere_sauvegarde > args.sauvegarde_min * 60:
                etat["secondes"] = secondes()
                etats.sauver("courant", model, optimizer, scheduler, scaler, etat)
                derniere_sauvegarde = time.time()
            if arret:
                fin_epoque = False
                break
        if not fin_epoque:
            break
        mean = total / max(1, n_steps)
        etat["historique"].append(mean)
        print("epoch %d/%d mean loss %.4f" % (epoch + 1, config.epochs, mean), flush=True)
        etat.update(epoch=epoch + 1, debut_lot=0, n_steps=0, total=0.0, prochaine_eval=1)
        arret = point_de_validation(epoch + 1, pas_par_epoque)
        arret = arret or (raison_de_s_arreter() if epoch + 1 < config.epochs else None)
        etat["secondes"] = secondes()
        if etats.sauver(f"epoque-{epoch + 1}", model, optimizer, scheduler, scaler, etat):
            etats.supprimer(etats.dossier / "courant.pt")
        derniere_sauvegarde = time.time()
        epoch += 1

    # Finalisation : poids du meilleur point de validation, calibrés sur cette même validation.
    from safetensors.torch import load_file
    poids = "courant"
    if etat["meilleur"] and (meilleur_dir / "model.safetensors").exists():
        model.load_state_dict({k: v.float() for k, v in load_file(str(meilleur_dir / "model.safetensors")).items()})
        poids = "meilleur"
    tous = [it for items in validation.values() for it in items]
    calib = records(model, tok, tous, dev, max_len, head_max_len, parallel, args.lot_validation) if tous else []
    effondrement = prior_collapse_message(calib) if calib else None
    if effondrement:
        warnings.warn(effondrement, RuntimeWarning)
    if calib:
        fitted = fit_temperature_map(calib)
        calibration = calibration_report(calib, fitted["temperature"])
        try:
            seuils = fit_abstention_thresholds(calib, temperature=fitted["temperature"],
                                               temperature_by_options=fitted["temperature_by_options"],
                                               target_error=config.target_error, min_bucket_n=config.min_abstain_n)
        except Exception as exc:  # noqa: BLE001 - même tolérance que laya.train.finetune
            warnings.warn(f"seuils d'abstention non ajustés : {exc}", RuntimeWarning)
            seuils = {}
    else:
        fitted, calibration, seuils = {"temperature": cfg.get("temperature", [1.0, 1.0, 1.0]),
                                       "temperature_by_options": {}}, {}, {}
    for nom, entree in calibration.items():
        for probleme in entree["issues"]:
            warnings.warn(f"calibrage {nom} : {probleme}", RuntimeWarning)

    session.update(fin=time.strftime("%Y-%m-%dT%H:%M:%S%z"), arret=arret, updates=updates_session)
    resultat = {
        "arret": arret, "epochs": config.epochs, "epoques_terminees": etat["epoch"], "updates": etat["updates"],
        "updates_prevus": total_updates, "secondes_session": round(time.time() - debut_session, 1),
        "secondes_total": round(secondes(), 1), "poids_finaux": poids, "meilleur": etat["meilleur"],
        "evaluations": etat["evaluations"], "historique": etat["historique"], "sessions": etat["sessions"],
        "validation": {"part": args.part_validation, "graine": args.graine_validation,
                       "cas": {t: len(v) for t, v in validation.items()}},
        "temperature": fitted["temperature"], "calibration": calibration, "effondrement": effondrement,
        "etat_reprise": str(etats.dossier), "pas_par_epoque": pas_par_epoque, "laya": LAYA_VERSION,
        "reglages": dict(asdict(config), part_validation=args.part_validation, evals_par_epoque=args.evals_par_epoque,
                         patience=args.patience, delta_min=args.delta_min, duree_max_h=args.duree_max_h),
    }
    out_cfg = dict(cfg, max_len=max_len, head_max_len=head_max_len, fine_tuned=True, temperature=fitted["temperature"])
    out_cfg.pop("temperature_by_options", None)
    if fitted["temperature_by_options"]:
        out_cfg["temperature_by_options"] = fitted["temperature_by_options"]
    # Les métadonnées d'entraînement de la base (mises à jour, heures…) ne décrivent pas ce
    # checkpoint : elles passent sous « base », et les clés lues par laya.evidence restent en place.
    out_cfg["training"] = {"laya_train": asdict(config), "laya_train_calibration": calibration,
                           "pilote": {k: v for k, v in resultat.items() if k not in ("evaluations", "calibration")},
                           "base": cfg.get("training")}
    if seuils:
        out_cfg["training"]["abstention_thresholds"] = seuils
    save_checkpoint(model, tok, out_cfg, args.out)
    schemas = {}
    for ligne in lignes:
        for qid, qdef in (ligne.get("questions") or {}).items():
            schemas.setdefault(qid, qdef)
    (Path(args.out) / "questions.json").write_text(json.dumps(schemas, ensure_ascii=False, indent=2), encoding="utf-8")
    Path(args.resultat).write_text(json.dumps(resultat, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"checkpoint {poids} calibré écrit dans {args.out} (températures {fitted['temperature']}) ; "
          f"arrêt : {arret or 'aucun'} ; {etat['epoch']} époque(s) terminée(s) sur {config.epochs}", flush=True)
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
if not PILOTE.exists():
    raise RuntimeError(f"{PILOTE} absent : exécuter la cellule précédente (%%writefile).")
PILOTE_SHA256 = sha256_file(PILOTE)
print("Pilote :", PILOTE, "sha256", PILOTE_SHA256[:16])


def commande_entrainement(verifier=False, reprendre=None):
    cmd = [sys.executable, str(PILOTE), "--data", str(TRAIN), "--meta", str(TRAIN_META), "--base", str(BASE_DIR),
           "--out", str(OUT_DIR), "--etats", str(ETATS_DIR), "--resultat", str(RESULTAT_TRAIN),
           "--epochs", str(EPOCHS), "--micro-batch", str(MICRO_BATCH), "--grad-accum", str(GRAD_ACCUM),
           "--encoder-lr", str(ENCODER_LR), "--head-lr", str(HEAD_LR), "--loss", LOSS,
           "--max-len", str(MAX_LEN), "--seed", str(SEED), "--device", "cuda",
           "--part-validation", str(PART_VALIDATION), "--evals-par-epoque", str(EVALS_PAR_EPOQUE),
           "--patience", str(PATIENCE), "--delta-min", str(DELTA_MIN), "--duree-max-h", str(DUREE_MAX_H),
           "--sauvegarde-min", str(SAUVEGARDE_MIN), "--garder-etats-epoque", str(GARDER_ETATS_EPOQUE)]
    if HEAD_MAX_LEN:
        cmd += ["--head-max-len", str(HEAD_MAX_LEN)]
    if reprendre is not None:
        cmd += ["--reprendre", str(reprendre)]
        if DOSSIER_REPRISE:
            cmd += ["--dossier-reprise", str(DOSSIER_REPRISE)]
    if verifier:
        cmd.append("--verifier")
    return cmd


_, arret = run_logged(commande_entrainement(verifier=True), LOG_DRY_RUN)
if arret:
    raise RuntimeError("Vérification interrompue : la relancer avant d'entraîner.")
budget = re.search(r"optimizer budget: (\d+) train item", LOG_DRY_RUN.read_text(encoding="utf-8"))
# La validation est retirée du jeu d'entraînement : le nombre d'exemples entraînés vient du pilote.
ITEMS_ENTRAINES = int(budget.group(1)) if budget else None
print("Exemples entraînés par époque :", ITEMS_ENTRAINES or "non lu dans le journal de vérification")

## 8. Évaluation **avant** entraînement (checkpoint de base)

Sur `laya-eval.jsonl`, avec le checkpoint de base, pour avoir le point de départ sur les mêmes cas que
l'« après ». Les cas sont regroupés par spécification de question (les consignes sont identiques au sein d'une
tâche) et envoyés par lots à `predict_batch`, comme le banc le fait en HTTP.

Métriques :
- **exactitude au seuil 0,5** : la décision binaire telle qu'un appelant l'utiliserait ;
- **AUROC** : qualité du classement, indépendante du seuil et du déséquilibre des classes ;
- pour `pertinence`, par question (parmi celles qui ont au moins un extrait pertinent) : **Hit@1** — l'extrait
  de plus forte probabilité est-il pertinent ? — et **Hit@7** — un pertinent figure-t-il dans les 7 premiers,
  ceux que le RAG garde ? C'est l'équivalent du « 1er rang » du banc. La ligne **ordre de la recherche**
  donne la même mesure sans reranker (ordre du fichier, celui de la recherche).
- **ex aequo** : laya arrondit la probabilité à 4 décimales. Un modèle saturé rend 0,0 pour presque tous les
  extraits, et plusieurs se partagent la première place. Hit@1 et Hit@7 sont alors comptés **en espérance**
  sur un départage au hasard, jamais dans l'ordre de la recherche, qui prêterait au modèle le tri de la
  recherche. `ex_aequo_en_tete` compte les questions concernées.
- **ordre appris** (`ordre_appris_hit_at_1`, `auroc_ordre_appris`) : seconde passe à température 50. La
  probabilité ne sature plus et garde l'ordre des logits : c'est ce que le modèle sait trier. Les autres
  lignes donnent ce que `laya-serve` rendrait au banc, à la température du checkpoint, arrondi compris.
  Un grand écart entre les deux signale un modèle qui trie bien mais que l'arrondi écrase : il faut
  alors le calibrer (le pilote le fait) ou lire ses logits, pas le jeter.

In [ ]:
import gc
import math
from collections import defaultdict   # aussi en section 5 : la cellule doit tourner seule après un redémarrage

import torch
from tqdm.auto import tqdm

import laya


def proba_oui(reponse, cle):
    """Probabilité du oui d'une question noul (même lecture que benchmarks/rag/systemone_client.py)."""
    reponse_cle = reponse["answers"][cle]
    valeur = reponse_cle.get("noul") if isinstance(reponse_cle, dict) else reponse_cle
    if valeur is None:
        raise KeyError(f"réponse noul absente pour {cle!r} : {reponse_cle!r}")
    return float(valeur)


def auroc(etiquettes, scores):
    """AUROC par la statistique de Mann-Whitney (rangs moyens en cas d'égalité) ; None si une classe manque."""
    n_pos = sum(etiquettes)
    n_neg = len(etiquettes) - n_pos
    if not n_pos or not n_neg:
        return None
    ordre = sorted(range(len(scores)), key=scores.__getitem__)
    rangs = [0.0] * len(scores)
    i = 0
    while i < len(ordre):
        j = i
        while j + 1 < len(ordre) and scores[ordre[j + 1]] == scores[ordre[i]]:
            j += 1
        for k in range(i, j + 1):
            rangs[ordre[k]] = (i + j) / 2 + 1
        i = j + 1
    somme_pos = sum(r for r, y in zip(rangs, etiquettes) if y)
    return (somme_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def predire(agent, etats, questions, taille_lot, max_len, titre):
    """predict_batch par lots ; divise le lot par deux en cas de manque de mémoire plutôt que d'échouer."""
    sorties, i = [], 0
    barre = tqdm(total=len(etats), desc=titre)
    while i < len(etats):
        lot = etats[i:i + taille_lot]
        try:
            with torch.no_grad():
                res = agent.predict_batch(lot, questions, batch_size=taille_lot, max_len=max_len)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if taille_lot == 1:
                raise
            taille_lot //= 2
            print(f"OOM : lot réduit à {taille_lot}")
            continue
        if len(res) != len(lot):
            raise RuntimeError(f"predict_batch a rendu {len(res)} résultats pour {len(lot)} états")
        sorties.extend(res)
        i += len(lot)
        barre.update(len(lot))
    barre.close()
    return sorties


def hits(par_question, score, k):
    """Part des questions dont l'un des k premiers (score décroissant) est pertinent, ex aequo départagés au hasard.

    Pourquoi l'espérance : laya arrondit la probabilité du oui à 4 décimales. Un modèle saturé rend
    0,0 pour la plupart des extraits ; départager ces ex aequo dans l'ordre de la recherche lui
    prêterait le tri de la recherche. Le checkpoint Kaggle du 26/11 (fin d'époque 1) affichait ainsi
    51 % au 1er rang, pour 26 % en espérance. Un groupe d'ex aequo à cheval sur la k-ième place compte
    pour la probabilité qu'un tirage au hasard de ses places y prenne un pertinent.
    """
    if not par_question:
        return None
    total = 0.0
    for v in par_question:
        restant, touche = k, 0.0
        for s in sorted({score(t) for t in v}, reverse=True):
            groupe = [t for t in v if score(t) == s]
            pertinents = sum(1 for t in groupe if t[2])
            if len(groupe) > restant:
                touche = 1.0 - math.comb(len(groupe) - pertinents, restant) / math.comb(len(groupe), restant)
                break
            if pertinents:
                touche = 1.0
                break
            restant -= len(groupe)
            if restant == 0:
                break
        total += touche
    return total / len(par_question)


def charger(chemin, repli_hub=False):
    try:
        return laya.load(str(chemin), device="cuda")
    except Exception as exc:  # noqa: BLE001
        if not repli_hub:
            raise
        # Repli : même révision, chargée par le Hub, si le chargement d'un chemin local est refusé.
        print(f"Chargement local refusé ({exc!r}) ; repli sur {REPO}@{REVISION[:8]}/{SUBFOLDER}")
        return laya.load(REPO, device="cuda", subfolder=SUBFOLDER, revision=REVISION)


# Ordre appris : à température 50, la probabilité du oui (sigmoïde de l'écart de logits / T) ne sature
# plus, et garde l'ordre des logits, monotone en T. Le checkpoint Kaggle du 26/11 (époque 1) rendait 0,0
# pour 1 520 extraits sur 1 573 à sa température ; à T = 50, son 1er rang passait de 26 % à 49 %.
# laya borne la température à 5 au chargement (common.TEMP_MAX), pas à l'usage : on la pose sur l'agent.
TEMPERATURE_ORDRE = 50.0


def passe(agent, cas, metas, titre):
    probas = [None] * len(cas)
    for tache, cle in CLE_PAR_TACHE.items():
        groupes = defaultdict(list)
        for idx, (c, m) in enumerate(zip(cas, metas)):
            if m["tache"] == tache:
                groupes[json.dumps(c["questions"], sort_keys=True, ensure_ascii=False)].append(idx)
        if len(groupes) != 1:
            print(f"Note : {len(groupes)} spécifications de question différentes pour {tache}")
        for spec, indices in groupes.items():
            res = predire(agent, [cas[i]["state"] for i in indices], json.loads(spec),
                          EVAL_BATCH[tache], EVAL_MAX_LEN[tache], f"{titre} · {tache}")
            for i, r in zip(indices, res):
                probas[i] = proba_oui(r, cle)
    return probas


def evaluer(chemin_modele, etiquette, repli_hub=False):
    cas, metas = lire_jsonl(EVAL), lire_jsonl(EVAL_META)
    agent = charger(chemin_modele, repli_hub)
    debut = time.time()
    # Deux passes : à la température du checkpoint (ce que laya-serve rendrait au banc, arrondi à
    # 4 décimales compris), puis à TEMPERATURE_ORDRE (l'ordre que le modèle a appris).
    probas = passe(agent, cas, metas, etiquette)
    temperature_servie = list(agent.temperature)
    agent.temperature = [TEMPERATURE_ORDRE] * len(agent.temperature)
    agent.temperature_by_options = {}
    ordre = passe(agent, cas, metas, f"{etiquette} · ordre appris")

    metriques = {}
    for tache, cle in CLE_PAR_TACHE.items():
        idx = [i for i, m in enumerate(metas) if m["tache"] == tache]
        if not idx:
            print(f"Aucun cas {tache} dans {EVAL.name}")
            continue
        y = [cas[i]["expected"][cle] for i in idx]
        p = [probas[i] for i in idx]
        mt = {"cas": len(idx), "positifs": sum(y),
              "exactitude_0_5": sum((pi >= SEUIL) == yi for pi, yi in zip(p, y)) / len(idx),
              "auroc": auroc(y, p), "auroc_ordre_appris": auroc(y, [ordre[i] for i in idx]),
              "proba_moyenne_positifs": sum(pi for pi, yi in zip(p, y) if yi) / max(sum(y), 1),
              "proba_moyenne_negatifs": sum(pi for pi, yi in zip(p, y) if not yi) / max(len(y) - sum(y), 1)}
        if tache == "pertinence":
            par_q = defaultdict(list)   # question_id -> [(rang de recherche, proba, pertinent)]
            for i in idx:
                qid = metas[i]["question_id"]
                par_q[qid].append((len(par_q[qid]), probas[i], cas[i]["expected"][cle], ordre[i]))
            eligibles = [v for v in par_q.values() if any(t[2] for t in v)]
            par_ordre = lambda t: t[3]           # noqa: E731 - ordre appris, sans saturation
            par_proba = lambda t: t[1]           # noqa: E731 - ex aequo possibles (arrondi de laya)
            par_recherche = lambda t: -t[0]      # noqa: E731 - rang de recherche : jamais d'ex aequo
            mt.update(questions=len(par_q), questions_avec_positif=len(eligibles),
                      hit_at_1=hits(eligibles, par_proba, 1), hit_at_7=hits(eligibles, par_proba, 7),
                      ordre_recherche_hit_at_1=hits(eligibles, par_recherche, 1),
                      ordre_recherche_hit_at_7=hits(eligibles, par_recherche, 7),
                      ordre_appris_hit_at_1=hits(eligibles, par_ordre, 1),
                      ordre_appris_hit_at_7=hits(eligibles, par_ordre, 7),
                      # Questions dont la probabilité la plus haute est partagée : signe d'un modèle saturé.
                      ex_aequo_en_tete=sum(1 for v in eligibles
                                           if sum(1 for t in v if t[1] == max(u[1] for u in v)) > 1),
                      probas_distinctes=len({probas[i] for i in idx}))
        metriques[tache] = mt

    resultat = {"etiquette": etiquette, "modele": str(chemin_modele), "laya": md.version("laya"),
                "eval": EVAL.name, "eval_sha256": META["fichiers"]["evaluation"]["sha256"],
                "eval_max_len": EVAL_MAX_LEN, "seuil": SEUIL, "duree_s": round(time.time() - debut, 1),
                "temperature_servie": temperature_servie, "temperature_ordre": TEMPERATURE_ORDRE,
                "metriques": metriques,
                "predictions": [{"tache": m["tache"], "question_id": m["question_id"],
                                 "attendu": c["expected"][CLE_PAR_TACHE[m["tache"]]], "proba": p, "ordre": o}
                                for c, m, p, o in zip(cas, metas, probas, ordre)]}
    # Rendre la mémoire GPU : l'entraînement tourne dans un autre processus et en a besoin.
    del agent
    gc.collect()
    torch.cuda.empty_cache()
    print(json.dumps(metriques, ensure_ascii=False, indent=1))
    print(f"Mémoire GPU encore réservée par le noyau : {torch.cuda.memory_reserved() / 2**20:.0f} Mio")
    return resultat


AVANT = evaluer(BASE_DIR, "avant", repli_hub=True)
(WORK / "eval-avant.json").write_text(json.dumps(AVANT, ensure_ascii=False, indent=1), encoding="utf-8")

## 9. Entraînement

Le pilote tourne dans un sous-processus : sa sortie s'affiche en direct et part, horodatée, dans
`laya-train.log` (complété, pas écrasé, lors d'une reprise), et toute sa mémoire GPU est rendue à la fin. Quelle
que soit la façon dont il s'arrête, il écrit dans `OUT_DIR` un checkpoint **calibré** : le meilleur point de
validation.

| Arrêt | Cause | Reprendre ? |
|---|---|---|
| aucun | toutes les époques faites | seulement pour en ajouter : `EPOCHS` plus grand, `REPRENDRE = "dernier"` |
| `arret_anticipe` | `PATIENCE` validations sans progrès | possible avec une patience plus grande |
| `budget` | `DUREE_MAX_H` dépassé | `REPRENDRE = "dernier"` |
| `interruption` | bouton *Interrupt* (le pilote finit sa fenêtre, sauvegarde et calibre) | `REPRENDRE = "dernier"` |

La durée, la configuration et le résumé du pilote (`train-resultat.json` : validations, meilleur point,
températures, sessions) sont recopiés dans `train-config.json`, pour qu'un résultat du banc puisse être relié à
un entraînement précis.

In [ ]:
if REPRENDRE is None:
    for dossier in (OUT_DIR, ETATS_DIR):
        if dossier.exists() and any(dossier.iterdir()):
            # Ne jamais mélanger deux entraînements : le checkpoint serait ambigu.
            raise RuntimeError(f"{dossier} n'est pas vide : REPRENDRE = 'dernier' pour continuer, "
                               "ou le supprimer pour repartir de zéro.")
elif DOSSIER_REPRISE and not LOG_TRAIN.exists():
    # Nouvelle session : le journal de la précédente continue ici, pour la courbe de perte.
    ancien = Path(DOSSIER_REPRISE).parent / LOG_TRAIN.name
    if ancien.exists():
        shutil.copy(ancien, LOG_TRAIN)

commande = commande_entrainement(reprendre=REPRENDRE)
# Allocateur du sous-processus : segments extensibles, moins de fragmentation de la VRAM, sauf réglage déjà présent.
env_entrainement = {}
if not (os.environ.get("PYTORCH_ALLOC_CONF") or os.environ.get("PYTORCH_CUDA_ALLOC_CONF")):
    env_entrainement["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
pas_par_epoque = -(-ITEMS_ENTRAINES // MICRO_BATCH) if ITEMS_ENTRAINES else None   # laya compte en micro-batchs
debut = time.strftime("%Y-%m-%dT%H:%M:%S%z")
lancement = time.time()
DUREE, ARRET_CELLULE = run_logged(commande, LOG_TRAIN, env_overrides=env_entrainement, ajouter=REPRENDRE is not None,
                                  filet_s=(DUREE_MAX_H + 1) * 3600, budget_s=DUREE_MAX_H * 3600,
                                  pas_par_epoque=pas_par_epoque)

if not RESULTAT_TRAIN.exists() or RESULTAT_TRAIN.stat().st_mtime < lancement:
    raise RuntimeError(f"Le pilote n'a pas écrit {RESULTAT_TRAIN.name} (arrêt : {ARRET_CELLULE}). Ses états de "
                       f"reprise sont dans {ETATS_DIR} : relancer avec REPRENDRE = 'dernier'.")
RESULTAT = json.loads(RESULTAT_TRAIN.read_text(encoding="utf-8"))
ARRET, EPOQUES_TERMINEES = RESULTAT["arret"], RESULTAT["epoques_terminees"]
print(f"\nArrêt : {ARRET or 'aucun'} ; {EPOQUES_TERMINEES} époque(s) terminée(s) sur {EPOCHS} ; "
      f"{RESULTAT['updates']} mises à jour sur {RESULTAT['updates_prevus']} ; {DUREE / 3600:.2f} h dans cette session")
if RESULTAT["meilleur"]:
    mp = RESULTAT["meilleur"]
    print(f"Meilleur point de validation : époque {mp['epoque']}, pas {mp['pas']}, AUROC moyenne {mp['score']:.4f}")
print("Poids finaux :", RESULTAT["poids_finaux"], "| températures calibrées :", RESULTAT["temperature"])
if ARRET in ("budget", "interruption"):
    print("Entraînement incomplet : pour le poursuivre, REPRENDRE = 'dernier' (dans une nouvelle session, "
          "DOSSIER_REPRISE vers les états de celle-ci), puis tout relancer.")
if RESULTAT.get("effondrement"):
    print("ATTENTION :", RESULTAT["effondrement"])

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
TRAIN_CONFIG = {
    "commande": commande, "debut": debut, "duree_s": round(DUREE, 1), "arret": ARRET,
    "epoques_terminees": EPOQUES_TERMINEES, "pilote_sha256": PILOTE_SHA256,
    "resultat": {k: RESULTAT.get(k) for k in ("updates", "updates_prevus", "meilleur", "poids_finaux", "temperature",
                                              "validation", "sessions", "secondes_total")},
    "laya": md.version("laya"), "torch": md.version("torch"), "gpu": gpu,
    "base": BASE_INFO, "donnees": {"dossier": str(DATA_DIR), "fichiers": META["fichiers"],
                                   "graine_decoupage": META.get("graine_decoupage"),
                                   "consignes_sha256": META.get("consignes_sha256")},
    "parametres": {"max_len": MAX_LEN, "head_max_len": HEAD_MAX_LEN, "micro_batch": MICRO_BATCH,
                   "grad_accum": GRAD_ACCUM, "epochs": EPOCHS, "encoder_lr": ENCODER_LR, "head_lr": HEAD_LR,
                   "loss": LOSS, "seed": SEED, "duree_max_h": DUREE_MAX_H, "part_validation": PART_VALIDATION,
                   "evals_par_epoque": EVALS_PAR_EPOQUE, "patience": PATIENCE, "delta_min": DELTA_MIN,
                   "reprendre": REPRENDRE, "dossier_reprise": str(DOSSIER_REPRISE) if DOSSIER_REPRISE else None},
}
(WORK / "train-config.json").write_text(json.dumps(TRAIN_CONFIG, ensure_ascii=False, indent=1), encoding="utf-8")

## 10. Évaluation **après** entraînement et comparaison

Même fonction, mêmes cas, mêmes longueurs : seul le checkpoint change. C'est celui que le pilote a écrit dans
`OUT_DIR` : le meilleur point de validation, calibré. `laya-eval` n'a servi ni à l'entraînement, ni à l'arrêt
anticipé, ni au calibrage : la mesure reste indépendante.

In [ ]:
CHECKPOINT = OUT_DIR
if not (CHECKPOINT / "rl_agent_config.json").exists():
    raise RuntimeError(f"Pas de checkpoint dans {CHECKPOINT} : l'entraînement (section 9) n'a pas abouti.")
if "RESULTAT" not in globals():   # après un redémarrage du noyau
    RESULTAT = json.loads(RESULTAT_TRAIN.read_text(encoding="utf-8"))
config_ckpt = json.loads((CHECKPOINT / "rl_agent_config.json").read_text(encoding="utf-8"))
CALIBRE = config_ckpt.get("fine_tuned") is True
print("Checkpoint évalué :", CHECKPOINT, "| calibré :", CALIBRE, "| températures :", config_ckpt.get("temperature"))
APRES = evaluer(CHECKPOINT, "après")
APRES["checkpoint"] = {"dossier": str(CHECKPOINT.relative_to(WORK)), "calibre": CALIBRE,
                       "temperature": config_ckpt.get("temperature"), "arret": RESULTAT["arret"],
                       "epoques_terminees": RESULTAT["epoques_terminees"], "epochs": RESULTAT["epochs"],
                       "poids": RESULTAT["poids_finaux"], "meilleur": RESULTAT["meilleur"]}
(WORK / "eval-apres.json").write_text(json.dumps(APRES, ensure_ascii=False, indent=1), encoding="utf-8")

import pandas as pd

lignes = []
for tache, mt in AVANT["metriques"].items():
    noms = ["exactitude_0_5", "auroc", "proba_moyenne_positifs", "proba_moyenne_negatifs"]
    if tache == "pertinence":
        noms += ["hit_at_1", "hit_at_7", "ordre_appris_hit_at_1", "ordre_appris_hit_at_7",
                 "ordre_recherche_hit_at_1", "ordre_recherche_hit_at_7", "ex_aequo_en_tete"]
    noms.insert(2, "auroc_ordre_appris")
    for nom in noms:
        a, b = mt.get(nom), APRES["metriques"].get(tache, {}).get(nom)
        lignes.append({"tâche": tache, "métrique": nom, "avant": a, "après": b,
                       "écart": None if a is None or b is None else b - a})
COMPARAISON = pd.DataFrame(lignes)
COMPARAISON.to_csv(WORK / "comparaison-avant-apres.csv", index=False)
(WORK / "comparaison-avant-apres.json").write_text(COMPARAISON.to_json(orient="records", force_ascii=False, indent=1),
                                                   encoding="utf-8")
# Les lignes « ordre_recherche » ne dépendent pas du modèle : elles restent identiques avant et après.
display(COMPARAISON.style.format({"avant": "{:.3f}", "après": "{:.3f}", "écart": "{:+.3f}"}, na_rep="—"))

## 11. Visualisation

Quatre figures, enregistrées dans `figures/` et jointes à l'archive :

1. **Métriques avant / après** : bon extrait au 1er rang (Hit@1), parmi les 7 transmis au LLM (Hit@7), AUROC et
   exactitude au seuil de 0,5. La ligne pointillée orange est l'**ordre de la recherche seule**, sans reranker :
   c'est la référence à battre, car un tri qui ferait moins bien ferait perdre des réponses.
2. **Distributions des probabilités** : un modèle utile pousse les extraits pertinents vers 1 et les autres vers 0 ;
   deux histogrammes superposés signifient qu'il ne distingue rien.
3. **Courbes ROC** : plus la courbe monte vite vers le coin supérieur gauche, mieux le modèle classe ; la diagonale
   est le hasard.
4. **Perte d'entraînement** : perte de chaque pas journalisé (un exemple : très bruitée), lissée, et moyenne de
   chaque époque.
5. **Validation pendant l'entraînement** : AUROC de chaque tâche à chaque validation, meilleur point retenu
   (étoile) et arrêt anticipé s'il a eu lieu. Si la courbe monte encore à la dernière époque, plus d'époques
   pourraient aider (`REPRENDRE = "dernier"` avec `EPOCHS` plus grand) ; si elle redescend, le modèle
   sur-apprend, et c'est le meilleur point, pas le dernier, qui est gardé.

Après un redémarrage du noyau, la cellule relit `eval-avant.json` et `eval-apres.json` : inutile de tout recalculer.

In [ ]:
import re

import matplotlib.pyplot as plt

FIG_DIR = WORK / "figures"
FIG_DIR.mkdir(exist_ok=True)
if "AVANT" not in globals():
    AVANT = json.loads((WORK / "eval-avant.json").read_text(encoding="utf-8"))
if "APRES" not in globals():
    APRES = json.loads((WORK / "eval-apres.json").read_text(encoding="utf-8"))

COULEURS = {"avant": "#9aa5b1", "après": "#12355b"}
REFERENCE = "#c2410c"
TACHES = ["pertinence", "reponse_possible"]
METRIQUES = {"pertinence": ["hit_at_1", "hit_at_7", "auroc", "exactitude_0_5"],
             "reponse_possible": ["auroc", "exactitude_0_5"]}
LIBELLES = {"hit_at_1": "1er rang", "hit_at_7": "dans les 7", "auroc": "AUROC", "exactitude_0_5": "exactitude (0,5)"}


def montrer(fig, nom):
    fig.tight_layout()
    fig.savefig(FIG_DIR / nom, dpi=130)
    plt.show()
    plt.close(fig)


# 1. Métriques avant / après, avec la référence « recherche seule ».
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), gridspec_kw={"width_ratios": [2, 1]})
for ax, tache in zip(axes, TACHES):
    noms = METRIQUES[tache]
    for decalage, (etiquette, res) in zip((-0.2, 0.2), (("avant", AVANT), ("après", APRES))):
        valeurs = [res["metriques"].get(tache, {}).get(n) for n in noms]
        barres = ax.bar([i + decalage for i in range(len(noms))], [v or 0 for v in valeurs], 0.4,
                        label=etiquette, color=COULEURS[etiquette])
        for barre, v in zip(barres, valeurs):
            ax.text(barre.get_x() + barre.get_width() / 2, (v or 0) + 0.02, "—" if v is None else f"{v:.2f}",
                    ha="center", fontsize=8)
    if tache == "pertinence":
        for nom, style in (("ordre_recherche_hit_at_1", "--"), ("ordre_recherche_hit_at_7", ":")):
            ref = AVANT["metriques"][tache].get(nom)
            if ref is not None:
                ax.axhline(ref, ls=style, lw=1, color=REFERENCE,
                           label=f"recherche seule, {'1er rang' if nom.endswith('1') else 'dans les 7'} = {ref:.2f}")
    ax.set_xticks(range(len(noms)))
    ax.set_xticklabels([LIBELLES[n] for n in noms])
    ax.set_ylim(0, 1.12)
    ax.set_title(tache)
    ax.legend(fontsize=8, loc="upper left", ncol=2)  # au-dessus des barres, sans les masquer
fig.suptitle("Laya sur laya-eval : avant / après entraînement")
montrer(fig, "1-metriques-avant-apres.png")


# 2. Distributions des probabilités du « oui », positifs contre négatifs.
def probas(res, tache, attendu):
    return [p["proba"] for p in res["predictions"] if p["tache"] == tache and p["attendu"] == attendu]


fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for col, tache in enumerate(TACHES):
    for ligne, (etiquette, res) in enumerate((("avant", AVANT), ("après", APRES))):
        ax = axes[ligne][col]
        ax.hist(probas(res, tache, False), bins=20, range=(0, 1), alpha=0.6, color=COULEURS["avant"], label="négatifs")
        ax.hist(probas(res, tache, True), bins=20, range=(0, 1), alpha=0.8, color=COULEURS["après"], label="positifs")
        ax.axvline(SEUIL, ls=":", color=REFERENCE)
        ax.set_title(f"{tache} — {etiquette}")
        ax.set_ylabel("cas")
        ax.legend(fontsize=8)
for ax in axes[1]:
    ax.set_xlabel("probabilité du oui")
montrer(fig, "2-distributions-probabilites.png")


# 3. Courbes ROC (ex aequo dans l'ordre d'origine : pas de tri favorable).
def courbe_roc(res, tache):
    points = sorted(((p["proba"], p["attendu"]) for p in res["predictions"] if p["tache"] == tache),
                    key=lambda t: -t[0])
    n_pos = sum(1 for _, y in points if y)
    n_neg = len(points) - n_pos
    if not n_pos or not n_neg:
        return None
    vp = fp = 0
    xs, ys = [0.0], [0.0]
    for _, y in points:
        vp, fp = vp + bool(y), fp + (not y)
        xs.append(fp / n_neg)
        ys.append(vp / n_pos)
    return xs, ys


fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, tache in zip(axes, TACHES):
    for etiquette, res in (("avant", AVANT), ("après", APRES)):
        courbe = courbe_roc(res, tache)
        if courbe:
            auc = res["metriques"][tache].get("auroc")
            ax.plot(*courbe, color=COULEURS[etiquette], label=f"{etiquette} (AUROC {auc:.3f})")
    ax.plot([0, 1], [0, 1], ls=":", color="grey", label="hasard")
    ax.set_xlabel("taux de faux positifs")
    ax.set_ylabel("taux de vrais positifs")
    ax.set_title(tache)
    ax.legend(fontsize=8, loc="lower right")
montrer(fig, "3-courbes-roc.png")

# 4. Perte d'entraînement. laya 0.4.22 écrit « epoch 1/4 step 100 loss 0.0196 » tous les 100 micro-batchs et
# « epoch 1/4 mean loss 2.1783 » en fin d'époque (laya/train.py, train_model). Avec un micro-batch de 1, la
# perte d'un pas ne porte que sur un exemple : très bruitée, elle est lissée par une moyenne glissante. Toute
# autre ligne « époque … perte » est lue comme une perte d'époque ; si rien n'est lisible, on le dit plutôt
# que d'inventer une courbe.
PAS_PERTE = re.compile(r"epoch (\d+)/\d+ step (\d+) loss (-?\d+(?:\.\d+)?(?:e-?\d+)?)")
NOMBRE = r"(-?\d+(?:\.\d+)?(?:e-?\d+)?)"
PERTES, PERTES_PAS = [], []
if LOG_TRAIN.exists():
    for ligne in LOG_TRAIN.read_text(encoding="utf-8", errors="replace").splitlines():
        if m := PAS_PERTE.search(ligne):
            PERTES_PAS.append((int(m.group(1)), int(m.group(2)), float(m.group(3))))
        elif m := re.search(r"epoch\D{0,3}(\d+)(?:/\d+)?.*?loss\D{0,3}" + NOMBRE, ligne, re.I):
            PERTES.append((int(m.group(1)), float(m.group(2))))
# Une reprise rejoue les pas faits depuis le dernier état : on garde la dernière occurrence de chacun.
PERTES_PAS = sorted({(e, s): (e, s, p) for e, s, p in PERTES_PAS}.values())
PERTES = sorted(dict(PERTES).items())
if PERTES or PERTES_PAS:
    fig, ax = plt.subplots(figsize=(8, 3.8))
    if PERTES_PAS:
        # Abscisse en époques : pas / nombre de pas d'une époque (le dernier pas journalisé d'une époque finie).
        pas_max = {}
        for e, s, _ in PERTES_PAS:
            pas_max[e] = max(pas_max.get(e, 0), s)
        longueur = max(pas_max.values())
        x = [e - 1 + s / longueur for e, s, _ in PERTES_PAS]
        y = [p for _, _, p in PERTES_PAS]
        fenetre = 10
        lisse = [sum(y[max(0, i - fenetre + 1):i + 1]) / len(y[max(0, i - fenetre + 1):i + 1]) for i in range(len(y))]
        ax.scatter(x, y, s=6, color=COULEURS["avant"], label="un pas (1 micro-batch)")
        ax.plot(x, lisse, color=COULEURS["après"], label=f"moyenne glissante ({fenetre} points)")
    if PERTES:
        ax.plot([e for e, _ in PERTES], [p for _, p in PERTES], marker="o", ls="--", color=REFERENCE,
                label="moyenne de l'époque (à sa fin)")
    ax.set_xlabel("époque")
    ax.set_ylabel("perte")
    ax.set_title("Perte d'entraînement (journal laya-train)")
    ax.legend(fontsize=8)
    montrer(fig, "4-perte-entrainement.png")
else:
    print(f"Aucune perte lisible dans {LOG_TRAIN.name} : figure 4 non produite (voir le journal brut).")
# 5. Validation pendant l'entraînement (train-resultat.json, écrit par le pilote).
if "RESULTAT" not in globals() and RESULTAT_TRAIN.exists():
    RESULTAT = json.loads(RESULTAT_TRAIN.read_text(encoding="utf-8"))
evaluations = (globals().get("RESULTAT") or {}).get("evaluations") or []
if evaluations:
    fig, ax = plt.subplots(figsize=(8, 3.8))
    x = [e["position"] for e in evaluations]
    for tache, style in zip(TACHES, ("-", "--")):
        y = [e["taches"].get(tache, {}).get("auroc") for e in evaluations]
        ax.plot(x, y, ls=style, marker="o", ms=4, color=COULEURS["après"], label=f"{tache} (AUROC)")
    ax.plot(x, [e["score"] for e in evaluations], color=REFERENCE, lw=1, label="score (moyenne)")
    mp = RESULTAT.get("meilleur")
    if mp:
        ax.scatter([mp["position"]], [mp["score"]], marker="*", s=180, color=REFERENCE, zorder=3,
                   label=f"meilleur point retenu ({mp['score']:.3f})")
    if RESULTAT.get("arret") == "arret_anticipe":
        ax.axvline(x[-1], ls=":", color="grey", label="arrêt anticipé")
    ax.set_xlabel("époque")
    ax.set_ylabel("AUROC (validation)")
    ax.set_title("Validation pendant l'entraînement (questions mises de côté, hors laya-eval)")
    ax.legend(fontsize=8)
    montrer(fig, "5-validation-entrainement.png")
else:
    print("Pas de validations dans train-resultat.json : figure 5 non produite.")
print("Figures :", sorted(p.name for p in FIG_DIR.glob("*.png")))

## 12. Conclusion

La cellule rédige le verdict à partir des chiffres de l'évaluation et l'enregistre (`conclusion.md`,
`conclusion.json`, joints à l'archive). **Règle de lecture fixée avant de voir les résultats :**

- **Prometteur** : après entraînement, le bon extrait arrive au 1er rang plus souvent qu'avec l'ordre de la
  recherche seule (la référence « sans reranker »), **et** l'AUROC progresse sur les deux tâches.
- **Insuffisant** : le 1er rang reste sous l'ordre de la recherche seule. Un tel modèle ferait perdre des
  réponses s'il triait les extraits : il ne doit pas aller au banc comme reranker.
- **Mitigé** : tous les autres cas ; à départager au banc.

Quel que soit le verdict, ce n'est qu'une indication : `laya-eval` compte peu de questions, issues du même
générateur que l'entraînement. **La décision se prend au banc**, sur les jeux `golden/` jamais vus ici.

In [ ]:
from IPython.display import Markdown, display

if "AVANT" not in globals():
    AVANT = json.loads((WORK / "eval-avant.json").read_text(encoding="utf-8"))
if "APRES" not in globals():
    APRES = json.loads((WORK / "eval-apres.json").read_text(encoding="utf-8"))


def val(res, tache, nom):
    return res["metriques"].get(tache, {}).get(nom)


def pct(v):
    return "—" if v is None else f"{100 * v:.1f} %".replace(".", ",")


def nb(v):
    return "—" if v is None else f"{v:.3f}".replace(".", ",")


def ecart(a, b, en_points=True):
    if a is None or b is None:
        return "—"
    return (f"{100 * (b - a):+.1f} pts" if en_points else f"{b - a:+.3f}").replace(".", ",")


h1 = (val(AVANT, "pertinence", "hit_at_1"), val(APRES, "pertinence", "hit_at_1"))
h7 = (val(AVANT, "pertinence", "hit_at_7"), val(APRES, "pertinence", "hit_at_7"))
ref1, ref7 = val(AVANT, "pertinence", "ordre_recherche_hit_at_1"), val(AVANT, "pertinence", "ordre_recherche_hit_at_7")
au_p = (val(AVANT, "pertinence", "auroc"), val(APRES, "pertinence", "auroc"))
au_r = (val(AVANT, "reponse_possible", "auroc"), val(APRES, "reponse_possible", "auroc"))
n_questions = val(APRES, "pertinence", "questions_avec_positif")
o1 = (val(AVANT, "pertinence", "ordre_appris_hit_at_1"), val(APRES, "pertinence", "ordre_appris_hit_at_1"))
au_o = (val(AVANT, "pertinence", "auroc_ordre_appris"), val(APRES, "pertinence", "auroc_ordre_appris"))

bat_recherche = None if h1[1] is None or ref1 is None else h1[1] > ref1
auroc_progresse = all(a is not None and b is not None and b > a for a, b in (au_p, au_r))
if bat_recherche and auroc_progresse:
    VERDICT = "prometteur"
    phrase = ("Après entraînement, Laya place le bon extrait en tête plus souvent que la recherche seule, et sépare "
              "mieux les deux tâches : il mérite d'être remesuré au banc comme reranker.")
elif bat_recherche is False:
    VERDICT = "insuffisant"
    phrase = ("Après entraînement, Laya place encore le bon extrait en tête moins souvent que la recherche seule : "
              "utilisé pour trier, il ferait perdre des réponses. Ne pas l'envoyer au banc comme reranker en l'état.")
else:
    VERDICT = "mitigé"
    phrase = "Les indicateurs ne vont pas tous dans le même sens : seul le banc, sur les jeux golden/, peut trancher."
# Le verdict porte sur ce que laya-serve rendrait (arrondi compris) ; l'ordre appris dit si l'écart vient
# de la saturation plutôt que du modèle.
if VERDICT != "prometteur" and o1[1] is not None and ref1 is not None and o1[1] > ref1:
    phrase += (" Sans l'arrondi de laya, son ordre appris bat pourtant la recherche seule : le calibrer, ou lire "
               "ses logits, avant de conclure.")

duree = f"{DUREE / 60:.0f} min" if "DUREE" in globals() else "non mesurée dans cette session"
ckpt = APRES.get("checkpoint") or {}
mp = ckpt.get("meilleur") or {}
point = (f"meilleur point de validation (époque {mp.get('epoque')}, pas {mp.get('pas')}, AUROC moyenne "
         f"{nb(mp.get('score'))})" if mp else "derniers poids (aucune validation)")
etat_entrainement = {
    None: f"Entraînement complet : {ckpt.get('epoques_terminees')} époque(s) sur {ckpt.get('epochs')}.",
    "arret_anticipe": (f"Arrêt anticipé après {ckpt.get('epoques_terminees')} époque(s) sur {ckpt.get('epochs')} : "
                       "la validation ne progressait plus."),
}.get(ckpt.get("arret"), f"**Réserve : entraînement incomplet** (arrêt : {ckpt.get('arret')}, "
                         f"{ckpt.get('epoques_terminees')} époque(s) sur {ckpt.get('epochs')}). Un entraînement "
                         "complet peut faire mieux ou moins bien : le reprendre avec `REPRENDRE = \"dernier\"`.")
reserve = (f"\n{etat_entrainement} Checkpoint évalué : {point}, "
           f"{'calibré sur la validation' if ckpt.get('calibre') else '**non calibré**'} "
           f"(températures {ckpt.get('temperature')}).\n") if ckpt else ""
ex_aequo = val(APRES, "pertinence", "ex_aequo_en_tete")
saturation = ""
if ex_aequo and n_questions and ex_aequo > 0.2 * n_questions:
    saturation = (f"\n**Probabilités saturées** : pour {ex_aequo} des {n_questions} questions, plusieurs extraits "
                  f"partagent la probabilité la plus haute ({val(APRES, 'pertinence', 'probas_distinctes')} valeurs "
                  "distinctes au total ; laya arrondit à 4 décimales). Le 1er rang est compté en espérance sur leur "
                  "départage : le modèle ne sépare pas ces extraits.\n")
TEXTE = f"""### Verdict : {VERDICT}

{phrase}
{reserve}{saturation}
| Indicateur (laya-eval, {n_questions if n_questions is not None else '—'} questions avec un extrait pertinent) | Avant | Après | Écart | Recherche seule |
|---|---:|---:|---:|---:|
| Pertinence : bon extrait au 1er rang | {pct(h1[0])} | {pct(h1[1])} | {ecart(*h1)} | {pct(ref1)} |
| Pertinence : bon extrait dans les 7 | {pct(h7[0])} | {pct(h7[1])} | {ecart(*h7)} | {pct(ref7)} |
| Pertinence : 1er rang, ordre appris (sans arrondi) | {pct(o1[0])} | {pct(o1[1])} | {ecart(*o1)} | {pct(ref1)} |
| Pertinence : AUROC | {nb(au_p[0])} | {nb(au_p[1])} | {ecart(*au_p, en_points=False)} | — |
| Pertinence : AUROC, ordre appris | {nb(au_o[0])} | {nb(au_o[1])} | {ecart(*au_o, en_points=False)} | — |
| Réponse possible : AUROC | {nb(au_r[0])} | {nb(au_r[1])} | {ecart(*au_r, en_points=False)} | — |

Entraînement : {duree}, `laya {md.version('laya')}`, base `{REPO}@{REVISION[:8]}/{SUBFOLDER}`.
Repères du banc R-54 (jeu v1, 197 questions) : 1er rang 41,1 % sans reranker, 46,2 % avec mmarco,
13,2 % avec Laya non entraîné.

### Suite

1. Télécharger l'archive (section 13) et vérifier son empreinte (`sha256sum -c`).
2. Servir le checkpoint à côté du Laya d'origine :
   `bash benchmarks/rag/laya/servir_checkpoint.sh <dossier-du-checkpoint> rag-fr`.
3. Remesurer au banc sur `golden/` v1 et v2 (`run_rag_benchmark.py --laya-url http://laya-bench-ft:8000 --laya-model rag-fr`),
   et comparer à mmarco et à la recherche seule.
4. Ne pas publier le checkpoint sans décision explicite.
"""
CONCLUSION = {"verdict": VERDICT, "phrase": phrase, "hit_at_1": h1, "hit_at_7": h7, "recherche_hit_at_1": ref1,
              "recherche_hit_at_7": ref7, "auroc_pertinence": au_p, "auroc_reponse_possible": au_r,
              "questions_avec_positif": n_questions, "entrainement": ckpt, "ordre_appris_hit_at_1": o1,
              "auroc_ordre_appris": au_o}
(WORK / "conclusion.md").write_text(TEXTE, encoding="utf-8")
(WORK / "conclusion.json").write_text(json.dumps(CONCLUSION, ensure_ascii=False, indent=1), encoding="utf-8")
display(Markdown(TEXTE))

## 13. Archivage pour téléchargement

Une seule archive à rapatrier : le checkpoint, les évaluations avant / après, les figures, la conclusion, les
journaux, la configuration et le pilote, avec un manifeste des empreintes de chaque fichier et l'empreinte de
l'archive elle-même (à recontrôler après téléchargement, avant de servir le modèle au banc).

Les **états de reprise** (`etats/`, ~3,7 Go chacun) ne vont pas dans l'archive : ils restent dans
`/kaggle/working`. Avec *Save & Run All*, ils font partie de la sortie de la version, que la session suivante
peut ajouter comme donnée d'entrée pour reprendre (`DOSSIER_REPRISE`).

In [ ]:
import zipfile

from IPython.display import FileLink

ANNEXES = ["eval-avant.json", "eval-apres.json", "comparaison-avant-apres.csv", "comparaison-avant-apres.json",
           "conclusion.md", "conclusion.json", "train-config.json", "train-resultat.json", "base-checkpoint.json",
           PILOTE.name,
           LOG_TRAIN.name, LOG_DRY_RUN.name]
contenu = [(p, f"{OUT_DIR.name}/{p.relative_to(OUT_DIR)}") for p in sorted(OUT_DIR.rglob("*")) if p.is_file()]
contenu += [(WORK / n, n) for n in ANNEXES if (WORK / n).exists()]
contenu += [(p, f"figures/{p.name}") for p in sorted((WORK / "figures").glob("*.png"))]
manifeste = {nom: {"octets": p.stat().st_size, "sha256": sha256_file(p)} for p, nom in contenu}

horodatage = time.strftime("%Y%m%d-%H%M%S")
ARCHIVE = WORK / f"laya-rag-fr-{horodatage}.zip"
with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for p, nom in contenu:
        zf.write(p, nom)
    zf.writestr("MANIFEST.json", json.dumps({"fichiers": manifeste, "checkpoint": str(CHECKPOINT.relative_to(WORK))},
                                            ensure_ascii=False, indent=1))

with zipfile.ZipFile(ARCHIVE) as zf:
    if zf.testzip() is not None:
        raise RuntimeError("Archive corrompue")
EMPREINTE = sha256_file(ARCHIVE)
(WORK / (ARCHIVE.name + ".sha256")).write_text(f"{EMPREINTE}  {ARCHIVE.name}\n", encoding="utf-8")
print(f"{ARCHIVE} : {ARCHIVE.stat().st_size / 2**20:.0f} Mio, {len(contenu) + 1} fichiers")
print("sha256 :", EMPREINTE)
display(FileLink(str(ARCHIVE.relative_to(WORK))))
etats = sorted(ETATS_DIR.glob("*.pt")) if ETATS_DIR.exists() else []
if etats:
    print(f"États de reprise laissés dans {ETATS_DIR} (hors archive) :",
          ", ".join(f"{p.name} {p.stat().st_size / 2**30:.1f} Gio" for p in etats))

## Après le téléchargement

1. Recontrôler l'empreinte : `sha256sum -c laya-rag-fr-<horodatage>.zip.sha256`.
2. Servir le checkpoint à côté du Laya d'origine, sans reconstruire d'image :
   `bash benchmarks/rag/laya/servir_checkpoint.sh <dossier> rag-fr` (voir `entrainement/README.md`).
3. Relancer `run_rag_benchmark.py --laya-url http://laya-bench-ft:8000 --laya-model rag-fr` sur les jeux
   `golden/` : c'est cette mesure, pas celle de ce notebook, qui dit si Laya entraîné vaut mieux que mmarco
   (46 % au 1er rang) ou que l'absence de reranker (41 %). Le banc refuse de mesurer si le serveur route vers un
   autre modèle que `rag-fr`.

**Ne pas publier le checkpoint** (Kaggle Models, Hugging Face, sortie d'un notebook public) sans décision
explicite : garder ce notebook et ses versions en privé. Le modèle de base est sous Apache-2.0, mais la
publication d'un dérivé engage Docaposte et se décide en dehors de ce notebook.